# Testing Pauli Propagation on Variational Benchmarks for Quantum Many-Body Problems

This notebook was used for the data evaluation of the master's thesis. 
All relevant data is in the folder thesis_data.
Under System Configurations, the .jld2 data of cluster or local runs can be loaded and the plotting settings are specified. 

# 1. Setup and Imports

In [ ]:
using JLD2
using PauliPropagation
using Plots
using Printf
using Statistics  # For mean() function
import Base: merge  # Explicitly import merge from Base to resolve ambiguity
using LaTeXStrings
# Load ADAPT modules
include("../adapt_modular.jl")

### Plotting Settings

In [ ]:
# === COLOR PALETTE FOR THESIS ===
using Colors

# bold palette
bold_hex = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#F0E442"]
bold = parse.(Colorant, bold_hex)

# Make pastel by mixing with white (tune α: 0.65–0.80 typically)
pastel(c; α=0.75) = RGB((1-α)*red(c)   + α*1.0,
                        (1-α)*green(c) + α*1.0,
                        (1-α)*blue(c)  + α*1.0)


pastels = [pastel(c; α=0.65) for c in bold]

# Usage:
# - Use bold[i] for line/marker in normal plots
# - Use pastels[i] for re-evaluated data (matching color to original)

In [ ]:
# Helper function to convert RGB color to hex string
function rgb_to_hex(c)
    r = round(Int, red(c) * 255)
    g = round(Int, green(c) * 255)
    b = round(Int, blue(c) * 255)
    return string("#", uppercase(string(r, base=16, pad=2)), 
                  uppercase(string(g, base=16, pad=2)), 
                  uppercase(string(b, base=16, pad=2)))
end

# Create hex versions of pastels
pastels_hex = [rgb_to_hex(c) for c in pastels]

println("Bold colors (hex): ", bold_hex)
println("Pastel colors (hex): ", pastels_hex)

In [ ]:
# === PLOTTING CONFIGURATION FOR THESIS ===
using Dates  # For timestamps in saved plots

pgfplotsx()

# --- Font sizes relative to 11pt document ---
const FS_TEXT   = 16  # match thesis text
const FS_LABEL  = 18
const FS_TICK   = 16
const FS_LEGEND = 18
const LW        = 2.0
const MS        = 5.0

# --- Thesis defaults ---
default(
    fontfamily = "Latin Modern Roman",   # closest to standard LaTeX serif
    titlefontsize = FS_TEXT, #not used
    guidefontsize = FS_LABEL,
    tickfontsize  = FS_TICK,
    legendfontsize = FS_LEGEND,
    linewidth = LW,
    markersize = MS,
    framestyle = :box,
    grid = true,
    foreground_color = :black,
    #background_color = :transparent,
    legend = :topright,
)
small = (600,400)
large = (900,600)

In [ ]:
# === HELPER FUNCTION FOR SAVING PLOTS ===
"""
    save_thesis_plot(p, title_text::String)

Helper function to save plots with timestamp and title-based filename.
The plot is saved to the plots_thesis/ directory as PDF.
"""
function save_thesis_plot(p, title_text::String)
    # Clean filename by replacing spaces with underscores and removing special characters
    clean_name = replace(title_text, " " => "_", r"[^a-zA-Z0-9_-]" => "")
    timestamp = Dates.format(now(), "yyyymmdd_HHMMSS")
    filename = "plots_thesis/$(clean_name)_$(timestamp).pdf"
    savefig(p, filename)
    println("  💾 Saved: $filename")
end

# Helper function to format MAC value as LaTeX exponential notation
function format_mac_latex(mac_value)
    if mac_value == 0
        return L"0"
    end
    exponent = floor(Int, log10(abs(mac_value)))
    coefficient = mac_value / 10.0^exponent
    if coefficient ≈ 1.0
        return latexstring("10^{$exponent}")
    else
        coeff_str = round(coefficient, digits=1)
        return latexstring("$(coeff_str) \\times 10^{$exponent}")
    end
end

# Helper function to format vector of MAC values as LaTeX string (returns plain string for use in labels)
function format_mac_vector_latex(mac_vector; as_latexstring=false, vector_brackets=true)
    if isempty(mac_vector)
        result = "[]"
        return as_latexstring ? latexstring(result) : result
    end
    
    # Format each element
    formatted_elements = String[]
    for mac_value in mac_vector
        if mac_value == 0
            push!(formatted_elements, "0")
        else
            exponent = floor(Int, log10(abs(mac_value)))
            coefficient = mac_value / 10.0^exponent
            if coefficient ≈ 1.0
                push!(formatted_elements, "10^{$exponent}")
            else
                coeff_str = round(coefficient, digits=1)
                push!(formatted_elements, "$(coeff_str) \\cdot 10^{$exponent}")
            end
        end
    end
    
    # Join with commas and wrap in brackets
    elements_str = join(formatted_elements, ", ")
    result = vector_brackets ? "[" * elements_str * "]" : elements_str
    return as_latexstring ? latexstring(result) : result
end

# 2. System Configuration

**Configure all system-specific details here:**
- Hamiltonian type using `SystemHamiltonian` type-based dispatch
- Topology type using `SystemTopology` type-based dispatch
- Overlap function
- ED and NQS reference data key
- Benchmark directory
- System parameters

**Note:** No target_max_weight and target_min_abs_coeff needed - we default to the last (most accurate) run saved.

In [ ]:
# Import custom PP integer types for large nq
_ = PauliSum(Float64, 35)
_ = PauliSum(Float64, 40)
_ = PauliSum(Float64, 45)
_ = PauliSum(Float64, 50)
_ = PauliSum(Float64, 55)
_ = PauliSum(Float64, 60)
_ = PauliSum(Float64, 70)
_ = PauliSum(Float64, 80)
_ = PauliSum(Float64, 90)
_ = PauliSum(Float64, 100)
import PauliPropagation: UInt72, UInt80, UInt96, UInt104, UInt120, UInt144

In [ ]:
# --- Hamiltonian Selection ---
# Options: "Heisenberg", "TFIM", "J1J2_1d", "J1J2_2d_obc", "tV", "Hubbard"
HAMILTONIAN_NAME = :J1J2_1d  

# --- Topology Selection ---
# Options: :chain, :chain_pbc, :square_obc
TOPOLOGY_NAME = :chain 

# --- System Parameters ---
tile_nq = 2       # Tile qubit count
min_nq = 20         # Minimum scaled qubit count
max_nq = 100      # Maximum scaled qubit count

# --- Initialize System Types ---
system_hamiltonian = SystemHamiltonian(HAMILTONIAN_NAME)
system_topology = SystemTopology(TOPOLOGY_NAME)

# Determine topology suffix for loading the master logs and the reference data abbreviation
if TOPOLOGY_NAME == :chain
    topology_suffix_load_master = "_chain_20"  # for 1D obc
    ed_ref_abbr = "obc"
elseif TOPOLOGY_NAME == :chain_pbc
    topology_suffix_load_master = "_chain_pbc"
    ed_ref_abbr = "pbc"
elseif TOPOLOGY_NAME == :square_obc
    topology_suffix_load_master = "_square_obc"
    ed_ref_abbr = "square_obc"
else
    error("Unsupported topology: $TOPOLOGY_NAME")
end


## Set data directories (folders of thesis_data)
All data can be found in the folder thesis_data, which has a number for which figure was generated with which data. For the file to run through, we need to provide comparison data.

In [ ]:
# scaled plot of the J1-J2 model up to 100 nq
benchmark_dir = "../thesis_data/Benchmark_SystemHamiltonian(:$(HAMILTONIAN_NAME))_forwarddiff_grad_(max_weight = Inf,)_tile_nq_2_rerun_fig3_1,3_2,5_1,5_10"
tile_nq = 2
method_legend_main = "Forward diff"

benchmark_dir_compare_1 = "../thesis_data/Benchmark_SystemHamiltonian(:J1J2_1d)_reversediff_grad_(max_weight = Inf,)_tile_nq_2_backend_comp_fig5_12,5_13"
tile_nq_compare_1 = 2
method_legend_compare_1 = " Reverse diff"

benchmark_dir_compare_2 ="../thesis_data/Benchmark_SystemHamiltonian(:$(HAMILTONIAN_NAME))_mooncake_grad_(max_weight = Inf,)_tile_nq_2_backend_comp_fig5_12,5_13"
tile_nq_compare_2 = 2
method_legend_compare_2 = " Mooncake"

In [ ]:
# --- Set Overlap Function and Hamiltonian Constructors Based on Hamiltonian ---
# Determine overlap function and create Hamiltonian constructor functions based on Hamiltonian type

# These constructor functions are needed for re-evaluation with different backends:
# - hamiltonian_constructor_simple(nq, topo): For Mooncake/ForwardDiff backends
# - hamiltonian_constructor_typed(CT, nq, topo): For ReverseDiff backend (requires type parameter)

if HAMILTONIAN_NAME == :Heisenberg
    # Simple constructor (non-typed)
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    # Typed constructor for ReverseDiff
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithneel
    
elseif HAMILTONIAN_NAME == :TFIM
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithplus
    
elseif HAMILTONIAN_NAME == :J1J2_1d
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithneel
    
elseif HAMILTONIAN_NAME == :J1J2_2d_obc
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithneel
    
elseif HAMILTONIAN_NAME == :tV
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithneel
    
elseif HAMILTONIAN_NAME == :Hubbard
    hamiltonian_constructor_simple = (nq, topo) -> get_hamiltonian(system_hamiltonian, nq, topo)
    hamiltonian_constructor_typed = (CT, nq, topo) -> get_hamiltonian(system_hamiltonian, CT, nq, topo)
    overlap_function = overlapwithneel
    
else
    error("Unknown HAMILTONIAN_NAME: $HAMILTONIAN_NAME")
end

# --- ED Data Configuration ---
# Build ED data key from Hamiltonian and topology
if HAMILTONIAN_NAME == :Heisenberg
    ed_data_key = "heisenberg_$(ed_ref_abbr)"
elseif HAMILTONIAN_NAME == :TFIM
    ed_data_key = "tfim_$(ed_ref_abbr)"
elseif HAMILTONIAN_NAME == :J1J2_1d
    ed_data_key = "j1j2_$(ed_ref_abbr)"
elseif HAMILTONIAN_NAME == :J1J2_2d_obc
    ed_data_key = "j1j2_2d_$(ed_ref_abbr)"
elseif HAMILTONIAN_NAME == :tV
    ed_data_key = "tv_$(ed_ref_abbr)"
elseif HAMILTONIAN_NAME == :Hubbard
    ed_data_key = "hubbard_$(ed_ref_abbr)"
else
    error("Unknown HAMILTONIAN_NAME: $HAMILTONIAN_NAME")
end

ed_data_file = "../reference_data/ed_reference_data/ed_reference_energies_20251120_102424.pkl" 
nqs_data_file = "../reference_data/nqs_reference_data/nqs_reference_energies_20251121_093613.pkl" 

# --- Re-evaluation Parameters ---
reeval_min_abs_coeff = 1e-10
reeval_max_weight = Inf
max_exact_reeval = 19  # Maximum nq for exact re-evaluation
println("="^80)
println("SYSTEM CONFIGURATION")
println("="^80)
println("Hamiltonian Type:    $system_hamiltonian")
println("Topology Type:       $system_topology")
println("ED Data Key:         $ed_data_key")
println("Benchmark Directory: $benchmark_dir")
println("Overlap Function:    $overlap_function")
println("Qubit Range:         $min_nq to $max_nq")
println("Max Exact Re-eval:   $max_exact_reeval qubits")
println("="^80);

In [ ]:
# Verify configuration is properly set up
println("\n" * "="^80)
println("CONFIGURATION VERIFICATION")
println("="^80)

# Test Hamiltonian constructor with type-based dispatch
test_nq = 4
test_topology = get_topology(system_topology, test_nq)
try
    # Test both versions (Float64 and parameterized)
    test_ham_simple = get_hamiltonian(system_hamiltonian, test_nq, test_topology)
    println("✓ Hamiltonian constructor (Float64) works: $(typeof(test_ham_simple))")
    
    test_ham_typed = get_hamiltonian(system_hamiltonian, Float64, test_nq, test_topology)
    println("✓ Hamiltonian constructor (CT-parameterized) works: $(typeof(test_ham_typed))")
catch e
    println("✗ Hamiltonian constructor failed: $e")
end

# Test topology construction
try
    println("✓ Topology constructor works: $(length(test_topology)) edges for $test_nq qubits")
catch e
    println("✗ Topology constructor failed: $e")
end

# Test overlap function
try
    println("✓ Overlap function: $overlap_function")
catch e
    println("✗ Overlap function error: $e")
end

# Check if directories exist
if isdir(benchmark_dir)
    println("✓ Benchmark directory exists: $benchmark_dir")
else
    println("⚠ Benchmark directory not found: $benchmark_dir")
end

if isfile(ed_data_file)
    println("✓ ED data file exists: $ed_data_file")
else
    println("⚠ ED data file not found: $ed_data_file")
end

println("="^80)
println("Configuration ready! Proceed to load data.\n");

# 3. Load Reference Data (ED and NQS)

In [ ]:
using Pickle # works for both python and julia, netket is in python which we used for the NQS benchmarks

# Load ED reference data using configured file
ed_data = Pickle.load(ed_data_file)

# Extract the relevant ED data for the configured Hamiltonian
ed_hamiltonian = ed_data[ed_data_key]

println("Loaded ED data for: $HAMILTONIAN_NAME")
println("Available ED qubit sizes: ", sort(collect(keys(ed_hamiltonian))))
println("Sample ED data structure: ", first(ed_hamiltonian))

# Load NQS reference data for larger system sizes (if available)
nqs_data = nothing
nqs_hamiltonian = Dict()
use_nqs_reference = false

if isfile(nqs_data_file)
    try
        nqs_data = Pickle.load(nqs_data_file)
        # Try to find matching key for NQS data
        # Could be same key as ED or different (e.g., "tfim_obc" vs "tfim_square_obc")
        if haskey(nqs_data, ed_data_key)
            nqs_hamiltonian = nqs_data[ed_data_key]
            use_nqs_reference = true
            println("\nLoaded NQS data for: $HAMILTONIAN_NAME (key: $ed_data_key)")
            println("Available NQS qubit sizes: ", sort(collect(keys(nqs_hamiltonian))))
            println("Sample NQS data structure: ", first(nqs_hamiltonian))
        else
            @warn "NQS data file found but does not contain key '$ed_data_key'. Available keys: $(keys(nqs_data))"
            println("Will use ED data as reference for all sizes.")
        end
    catch e
        @warn "Could not load NQS data: $e"
        println("Will use ED data as reference for all sizes.")
    end
else
    @warn "NQS data file not found: $nqs_data_file"
    println("Will use ED data as reference for all sizes.")
end

# Create combined reference data dictionary
# Prioritize ED data when available, use NQS for larger sizes (if NQS exists)
reference_energies = Dict{Int, Dict{String, Any}}()

# Add ED data
for (nq, data) in ed_hamiltonian
    reference_energies[nq] = Dict(
        "gs_per_site" => data["gs_per_site"],
        "ground_state" => data["ground_state"],
        "method" => "ED",
        "std_error" => 0.0  # ED is exact
    )
end

# Add NQS data only if it exists and was loaded successfully
if use_nqs_reference
    for (nq, data) in nqs_hamiltonian
        if !haskey(reference_energies, nq)  # Only add if not already in ED
            reference_energies[nq] = Dict(
                "gs_per_site" => data["gs_per_site"],
                "ground_state" => data["ground_state"],
                "method" => "NQS",
                "std_error" => data["std_error"]
            )
        end
    end
end

println("\n" * "="^80)
println("COMBINED REFERENCE DATA")
println("="^80)
println("Total qubit sizes available: ", sort(collect(keys(reference_energies))))
ed_sizes = sort([nq for (nq, data) in reference_energies if data["method"] == "ED"])
nqs_sizes = sort([nq for (nq, data) in reference_energies if data["method"] == "NQS"])
println("ED sizes: ", ed_sizes)
println("NQS sizes: ", nqs_sizes)
if !use_nqs_reference
    println("\nNote: Using ED as reference for all system sizes (no NQS data loaded)")
end
println("="^80);


In [ ]:
# Display reference data summary
println("\n" * "="^80)
println("REFERENCE DATA SUMMARY")
println("="^80)
@printf "%-8s %-20s %-20s %-12s\n" "Qubits" "Energy per Site" "Std Error" "Method"
println("-"^80)

for nq in sort(collect(keys(reference_energies)))
    data = reference_energies[nq]
    @printf "%-8d %-20.12f %-20.12f %-12s\n" nq data["gs_per_site"] data["std_error"] data["method"]
end

println("="^80)
println("\nSummary:")
ed_count = count(d["method"] == "ED" for d in values(reference_energies))
nqs_count = count(d["method"] == "NQS" for d in values(reference_energies))
println("  ED reference points: $ed_count")
println("  NQS reference points: $nqs_count")
println("  Total reference points: $(ed_count + nqs_count)")
println("="^80);

# 4. Load Multiple Loop Master Logs

Load loop master logs across all qubit counts. The looped data structure has:
- **Top-level**: `all_results` - Array of different truncation sequence runs
- **Each run contains**:
  - `sequence_metadata` - Information about MW/MAC sequences used
  - `loop_summary` - Combined results from entire progressive truncation sequence
  - `[1], [2], ...` - Individual sub-runs for each truncation level

**Default behavior**: Use the last saved run (most accurate truncation parameters)

In [ ]:
"""
    load_loop_master_logs_all_nq(dir_path; hamiltonian_name="Heisenberg", 
                                  topology_name="chain",
                                  tile_nq=3, min_nq=4, max_nq=16, 
                                  use_most_accurate=true)

Load loop master log files with new "all_results" structure for ALL qubit counts.

Loop files have structure:
- `data["all_results"]`: Array of loop runs, each with different truncation sequences
- Each run contains:
  - `["loop_summary"]`: Combined results from all truncation levels
  - `["sequence_metadata"]`: The specific truncation sequence used
  - `[1], [2], ...`: Individual sub-runs for each truncation level

# Arguments
- `dir_path`: Directory containing the JLD2 files
- `hamiltonian_name`: Name of Hamiltonian (e.g., "Heisenberg", "J1J2", "TFIM")
- `topology_name`: Topology name (e.g., "chain", "chain_pbc", "square_obc")
- `tile_nq`: Tile qubit count for filtering
- `min_nq`: Minimum scaled qubit count
- `max_nq`: Maximum scaled qubit count
- `use_most_accurate`: If true, use `all_results[end]` (default: true)

# Returns
- Dictionary mapping scaled_nq => loop_summary_dict (single dict, not array)

# New File Structure
Files are now named: `master_log_loop_SystemHamiltonian(:Name)_SystemTopology(:topology)_timestamp_tileN_scaledM.jld2`
Example: `master_log_loop_SystemHamiltonian(:Heisenberg)_SystemTopology(:chain)_20251228_163729_tile3_scaled5.jld2`
"""
function load_loop_master_logs_all_nq(dir_path::String; 
                                      hamiltonian_name::String="Heisenberg",
                                      topology_name::String="chain",
                                      tile_nq::Int=3,
                                      min_nq::Int=4,
                                      max_nq::Int=16,
                                      use_most_accurate::Bool=true)
    # Get all JLD2 files in directory
    all_files = filter(f -> endswith(f, ".jld2"), readdir(dir_path))
    
    # Filter specifically for loop runs with new structure:
    # master_log_loop_SystemHamiltonian(:Name)_SystemTopology(:topology)_...
    master_logs = filter(all_files) do f
        startswith(f, "master_log_loop_SystemHamiltonian(:$(hamiltonian_name))_SystemTopology(:$(topology_name))")
    end
    
    # Dictionary to store runs by scaled_nq
    runs_by_nq = Dict{Int, Dict{String, Any}}()
    
    println("Searching for LOOP master logs in: $dir_path")
    println("Pattern: master_log_loop_SystemHamiltonian(:$(hamiltonian_name))_SystemTopology(:$(topology_name))_*")
    println("Found $(length(master_logs)) loop master log files\n")
    
    for filename in master_logs
        # Try to extract scaled_nq from filename
        # Pattern: master_log_loop_SystemHamiltonian(:Name)_SystemTopology(:topology)_timestamp_tileX_scaledY.jld2
        if occursin(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
            m = match(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
            file_tile_nq = parse(Int, m.captures[1])
            file_scaled_nq = parse(Int, m.captures[2])
            
            # Only load if it matches our tile_nq and is in the qubit range
            if file_tile_nq == tile_nq && min_nq <= file_scaled_nq <= max_nq
                filepath = joinpath(dir_path, filename)
                
                try
                    # Load the file directly
                    data = JLD2.load(filepath)
                    
                    # Check for new "all_results" structure
                    if haskey(data, "all_results")
                        all_results = data["all_results"]
                        
                        # Default to most accurate run (last one)
                        run_idx = use_most_accurate ? length(all_results) : 1
                        selected_run = all_results[run_idx]
                        
                        # Extract loop_summary
                        if haskey(selected_run, "loop_summary")
                            loop_summary = selected_run["loop_summary"]
                            
                            # Store the loop_summary directly (not as array)
                            runs_by_nq[file_scaled_nq] = loop_summary
                            
                            # Print info about selected run
                            if haskey(selected_run, "sequence_metadata")
                                seq_meta = selected_run["sequence_metadata"]
                                println("✓ Loaded $filename: run $run_idx/$(length(all_results)) for $(file_scaled_nq) qubits")
                                println("  Sequence: mw=$(seq_meta["mw_sequence"]), mac=$(seq_meta["mac_sequence"])")
                            else
                                println("✓ Loaded $filename: run $run_idx/$(length(all_results)) for $(file_scaled_nq) qubits")
                            end
                        else
                            @warn "No loop_summary found in $filename run $run_idx"
                        end
                    else
                        @warn "No all_results structure found in $filename"
                    end
                catch e
                    @warn "Failed to load $filename: $e"
                end
            end
        end
    end
    
    # Sort by qubit count and print summary
    sorted_nqs = sort(collect(keys(runs_by_nq)))
    println("\n" * "="^80)
    println("Summary: Loaded loop data for $(length(sorted_nqs)) different system sizes")
    println("="^80)
    for nq in sorted_nqs
        loop_data = runs_by_nq[nq]
        println("  $nq qubits: Energy/Q=$(loop_data["final_energy_per_qubit"]), Depth=$(loop_data["final_circuit_depth"])")
    end
    println("="^80)
    
    return runs_by_nq
end


In [ ]:
# Load loop master logs using configured parameters

hamiltonian_loop_runs = load_loop_master_logs_all_nq(
    benchmark_dir; 
    hamiltonian_name=String(HAMILTONIAN_NAME),
    topology_name=String(TOPOLOGY_NAME),
    tile_nq=tile_nq,
    min_nq=min_nq,
    max_nq=max_nq,
    use_most_accurate=true  # Use the last (most accurate) saved run
)

println("\nLoaded loop benchmark runs for $HAMILTONIAN_NAME with topology $TOPOLOGY_NAME")
println("Qubit sizes available: ", sort(collect(keys(hamiltonian_loop_runs))))

# 5. Example: Accessing Loaded Data

The loaded data structure is now a dictionary mapping `nq => loop_summary_dict`.

Each `loop_summary_dict` contains:
- `final_energy_per_qubit`: Final energy per qubit from the full progressive truncation
- `final_circuit_depth`: Final circuit depth
- `combined_energy_curve`: Complete energy trajectory across all truncation levels
- `combined_max_grads`: Complete gradient trajectory
- `total_elapsed_time_s`: Total runtime

In [ ]:
# Display summary of all loaded runs
println("\n" * "="^80)
println("LOOP RUNS SUMMARY")
println("="^80)
@printf "%-8s %-20s %-15s %-15s %-15s\n" "Qubits" "Energy/Q" "Depth" "Time (s)" "Ref Method"
println("-"^80)

for nq in sort(collect(keys(hamiltonian_loop_runs)))
    loop_data = hamiltonian_loop_runs[nq]
    
    energy_per_q = loop_data["final_energy_per_qubit"]
    depth = loop_data["final_circuit_depth"]
    time_s = loop_data["total_elapsed_time_s"]
    # Get reference method if available
    ref_method = haskey(reference_energies, nq) ? reference_energies[nq]["method"] : "N/A"
    
    @printf "%-8d %-20.12f %-15d %-15.2f %-15s\n" nq energy_per_q depth time_s ref_method
end

println("="^80);

In [ ]:
# Example: Access specific qubit count data
example_nq = sort(collect(keys(hamiltonian_loop_runs)))[1]  # Pick first available

println("\n" * "="^80)
println("EXAMPLE: Accessing data for $example_nq qubits")
println("="^80)

if haskey(hamiltonian_loop_runs, example_nq)
    loop_data = hamiltonian_loop_runs[example_nq]
    
    println("\nLoop Summary Keys:")
    for key in sort(collect(keys(loop_data)))
        println("  - $key")
    end
    
    println("\nKey Metrics:")
    println("  Final Energy/Q: $(loop_data["final_energy_per_qubit"])")
    println("  Final Depth: $(loop_data["final_circuit_depth"])")
    println("  Total Time: $(round(loop_data["total_elapsed_time_s"], digits=2))s")
    println("  Energy curve length: $(length(loop_data["combined_energy_curve"])) iterations")
    
    if haskey(reference_energies, example_nq)
        ref_energy = reference_energies[example_nq]["ground_state"] / example_nq
        energy_diff = loop_data["final_energy_per_qubit"] - ref_energy
        println("  Reference Energy/Q: $ref_energy ($(reference_energies[example_nq]["method"]))")
        println("  Energy Difference: $energy_diff")
    end
    
else
    println("No data found for $example_nq qubits")
end

println("="^80);

In [ ]:
# Example: Compare multiple system sizes
println("\n" * "="^80)
println("ENERGY ACCURACY COMPARISON")
println("="^80)
@printf "%-8s %-20s %-20s %-20s\n" "Qubits" "ADAPT Energy/Q" "Reference Energy/Q" "Difference"
println("-"^80)

for nq in sort(collect(keys(hamiltonian_loop_runs)))
    if haskey(reference_energies, nq)
        adapt_energy = hamiltonian_loop_runs[nq]["final_energy_per_qubit"]
        ref_energy = reference_energies[nq]["ground_state"] / nq
        diff = adapt_energy - ref_energy
        
        @printf "%-8d %-20.12f %-20.12f %-20.12e\n" nq adapt_energy ref_energy diff
    end
end

println("="^80);

# 6. Data Evaluation

## Re-Evaluation Plot (Relative Accuracy with re-evaluations) 

In [ ]:
"""
    run_lossfunction(run, circuit, thetas, nq, hamiltonian_or_constructor, overlap_func; 
                     min_abs_coeff=0.0, max_weight=Inf, max_freq=Inf, topology=nothing, overlap_kwargs...)

Select and evaluate the correct loss function based on run configuration (backend and threads).

# Arguments
- `run`: Dictionary containing benchmark run metadata (must have "backend" and "threads" keys)
- `circuit`: Quantum circuit
- `thetas`: Circuit parameters
- `nq`: Number of qubits
- `hamiltonian_or_constructor`: Either a Hamiltonian (PauliSum) for MC/FD, or a constructor function for RD
- `overlap_func`: Function to compute overlap with initial state
- `min_abs_coeff`: Minimum absolute coefficient for truncation (MC/FD)
- `max_weight`: Maximum Pauli weight
- `max_freq`: Maximum frequency for truncation (RD)
- `topology`: System topology (required for RD)
- `overlap_kwargs`: Additional keyword arguments for overlap function

# Returns
- Energy expectation value using the appropriate loss function

# Notes
- For threads=false: Uses fulllossfunction_MC, _RD, or _FD based on backend
- For threads=true with MC or FD: Uses indiv_term_lossfunction_MC (same for both)
- For threads=true with RD: Uses indiv_term_lossfunction_RD
"""
function run_lossfunction(run, circuit, thetas, nq, hamiltonian_or_constructor, overlap_func; 
                          min_abs_coeff=0.0, max_weight=Inf, max_freq=Inf, topology=nothing, overlap_kwargs...)
    
    # Extract backend and threads from run metadata
    backend = Symbol(run["backend"])
    threads = run["threads"]
    
    println("  Backend: $backend, Threads: $threads")
    
    # Select appropriate loss function based on configuration
    if threads
        # threads=true: Use individual term loss functions with manual summation
        if backend == :reversediff
            # ReverseDiff with threads: use indiv_term_lossfunction_RD
            # hamiltonian_or_constructor should be a constructor function
            ham_constructor = hamiltonian_or_constructor
            hamiltonian = ham_constructor(Float64, nq, topology)
            num_terms = length(topaulistrings(hamiltonian))
            
            # Compute energy by summing over all Hamiltonian terms
            energy = 0.0
            for which in 1:num_terms
                term_energy = indiv_term_lossfunction_RD(thetas, which, circuit, nq, ham_constructor, overlap_func;
                                                          topology=topology, max_freq=max_freq, max_weight=max_weight, overlap_kwargs...)
                energy += term_energy
            end
            return energy
            
        else  # backend == :mooncake or :forwarddiff
            # Both Mooncake and ForwardDiff with threads use indiv_term_lossfunction_MC
            # hamiltonian_or_constructor should be a Hamiltonian (PauliSum)
            hamiltonian = hamiltonian_or_constructor
            num_terms = length(topaulistrings(hamiltonian))
            
            # Compute energy by summing over all Hamiltonian terms
            energy = 0.0
            for which in 1:num_terms
                term_energy = indiv_term_lossfunction_MC(thetas, which, circuit, nq, hamiltonian, overlap_func;
                                                          min_abs_coeff=min_abs_coeff, max_weight=max_weight, overlap_kwargs...)
                energy += term_energy
            end
            return energy
        end
        
    else
        # threads=false: Use full loss functions
        if backend == :mooncake
            # Mooncake: use fulllossfunction_MC
            hamiltonian = hamiltonian_or_constructor
            return fulllossfunction_MC(thetas, circuit, nq, hamiltonian, overlap_func;
                                       min_abs_coeff=min_abs_coeff, max_weight=max_weight, overlap_kwargs...)
            
        elseif backend == :forwarddiff
            # ForwardDiff: use fulllossfunction_FD
            hamiltonian = hamiltonian_or_constructor
            return fulllossfunction_FD(thetas, circuit, nq, hamiltonian, overlap_func;
                                       min_abs_coeff=min_abs_coeff, max_weight=max_weight, overlap_kwargs...)
            
        else  # backend == :reversediff
            # ReverseDiff: use fulllossfunction_RD
            ham_constructor = hamiltonian_or_constructor
            return fulllossfunction_RD(thetas, circuit, nq, ham_constructor, overlap_func;
                                       topology=topology, max_freq=max_freq, max_weight=max_weight, overlap_kwargs...)
        end
    end
end

In [ ]:
"""
    get_topology_from_run(run; default_topology=TOPOLOGY_NAME)

Reconstruct topology from run data using type-based dispatch.

# Arguments
- `run`: Dictionary containing benchmark run metadata
- `default_topology`: Default topology Symbol if not found in run data (default: global TOPOLOGY_NAME)

# Returns
- Topology object (e.g., from get_topology)

# Notes
- NEW: Checks for "topology_name" (new format with SystemTopology)
- OLD: Falls back to "topology_abbr" (old format)
- If neither exists: Uses default_topology (typically from global TOPOLOGY_NAME)
"""
function get_topology_from_run(run; default_topology=TOPOLOGY_NAME)
    nq = run["scaled_nq"]
    
    # Check for new format first: "topology_name" (stores Symbol)
    if haskey(run, "topology_name")
        topology_symbol = Symbol(run["topology_name"])
    # Fall back to old format: "topology_abbr" (stores string abbreviation)
    elseif haskey(run, "topology_abbr")
        topology_abbr = run["topology_abbr"]
        # Convert string abbreviation to Symbol
        if topology_abbr == "chain"
            topology_symbol = :chain
        elseif topology_abbr == "chain_pbc"
            topology_symbol = :chain_pbc
        elseif topology_abbr == "square_obc"
            topology_symbol = :square_obc
        else
            @warn "Unknown topology_abbr: $topology_abbr, using default: $default_topology"
            topology_symbol = default_topology
        end
    else
        # Neither format found - use default from global configuration
        topology_symbol = default_topology
    end
    
    # Use type-based dispatch to get topology
    topo_type = SystemTopology(topology_symbol)
    return get_topology(topo_type, nq)
end

In [ ]:
# Re-evaluate ALL qubit sizes with set truncations
# For loop runs, we need to load the original data files to get backend info from subruns
println("\n" * "="^90)
println("RE-EVALUATION WITH MATCHING LOSS FUNCTIONS (All Qubit Sizes)")
println("="^90)

# First, we need to reload the original files to access backend info from subruns
# because loop_summary doesn't contain backend information
original_data_by_nq = Dict{Int, Any}()

# Get all JLD2 files in directory
all_files = filter(f -> endswith(f, ".jld2"), readdir(benchmark_dir))

# Filter for new file structure: master_log_loop_SystemHamiltonian(:Name)_SystemTopology(:topology)_...
master_logs = filter(all_files) do f
    startswith(f, "master_log_loop_SystemHamiltonian(:$(HAMILTONIAN_NAME))_SystemTopology(:$(TOPOLOGY_NAME))")
end

println("Loading original files to extract backend information from subruns...")
for filename in master_logs
    # Extract scaled_nq from filename
    if occursin(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
        m = match(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
        file_tile_nq = parse(Int, m.captures[1])
        file_scaled_nq = parse(Int, m.captures[2])
        
        # Only load if it matches our tile_nq and is in our loaded data
        if file_tile_nq == tile_nq && haskey(hamiltonian_loop_runs, file_scaled_nq)
            filepath = joinpath(benchmark_dir, filename)
            
            try
                data = JLD2.load(filepath)
                if haskey(data, "all_results")
                    # Store the full data structure (we'll use the last run, matching our loop_summary)
                    original_data_by_nq[file_scaled_nq] = data
                    println("✓ Loaded original data for $file_scaled_nq qubits")
                end
            catch e
                @warn "Failed to load original data for $filename: $e"
            end
        end
    end
end

println("\nLoaded original data for $(length(original_data_by_nq)) system sizes")

# Extract initial mac and mw from the first subrun (initial parameters)
initial_mac = nothing
initial_mw = nothing

if !isempty(original_data_by_nq)
    # Get the first available nq to extract initial parameters
    first_nq = first(sort(collect(keys(original_data_by_nq))))
    original_data = original_data_by_nq[first_nq]
    all_results = original_data["all_results"]
    
    # Use the last run (most accurate) - all_results is an Array
    selected_run = all_results[length(all_results)]
    
    # Get first subrun to extract initial parameters
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    if !isempty(subrun_keys)
        first_subrun = selected_run[subrun_keys[1]]
        initial_mac = first_subrun["min_abs_coeff"]
        initial_mw = first_subrun["max_weight"]
        println("\nExtracted initial parameters from first subrun of $first_nq qubits:")
        println("  initial_mac = $initial_mac")
        println("  initial_mw = $initial_mw")
    else
        @warn "No subruns found to extract initial parameters"
    end
else
    @warn "No original data available to extract initial parameters"
end

println("="^90)

In [ ]:
# Perform re-evaluation for all loaded qubit sizes
reevaluated_energies = Dict{Int, Float64}()
reevaluated_energies_exact = Dict{Int, Float64}()  # Exact re-evaluation for ED systems
qubit_sizes_full = sort(collect(keys(hamiltonian_loop_runs)))
absolute_errors_full = Float64[]
relative_distances_full = Float64[]
below_gs_indices = Int[]

for (idx, nq) in enumerate(qubit_sizes_full)
    loop_summary = hamiltonian_loop_runs[nq]
    
    println("\n--- $nq Qubits ---")
    println("Original ADAPT energy/qubit: $(loop_summary["final_energy_per_qubit"])")
    
    # Get reference energy
    if !haskey(reference_energies, nq)
        @warn "No reference energy for nq=$nq, skipping"
        push!(absolute_errors_full, NaN)
        push!(relative_distances_full, NaN)
        continue
    end
    
    exact_energy = reference_energies[nq]["gs_per_site"]
    method = reference_energies[nq]["method"]
    
    # Calculate original errors
    abs_error = loop_summary["final_energy_per_qubit"] - exact_energy
    rel_error = abs(abs_error / exact_energy)
    push!(absolute_errors_full, abs_error)
    push!(relative_distances_full, rel_error)
    
    # Track if original was below GS
    if abs_error < 0
        push!(below_gs_indices, idx)
    end
    
    # Get backend info from original data (from first subrun)
    if !haskey(original_data_by_nq, nq)
        @warn "No original data found for nq=$nq, skipping re-evaluation"
        continue
    end
    
    original_data = original_data_by_nq[nq]
    all_results = original_data["all_results"]
    
    # Use the last run (most accurate) to match our loop_summary
    selected_run_idx = length(all_results)
    selected_run = all_results[selected_run_idx]
    
    # Get backend info from first subrun
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    if isempty(subrun_keys)
        @warn "No subruns found in original data for nq=$nq"
        continue
    end
    
    first_subrun = selected_run[subrun_keys[1]]
    backend = Symbol(first_subrun["backend"])
    threads = first_subrun["threads"]
    
    println("Backend from subrun: $backend, Threads: $threads")
    
    # Get topology from run data using type-based dispatch
    topo_scaled = get_topology_from_run(first_subrun)
    
    # Extract circuit directly from loop_summary
    circuit = loop_summary["circuit"]
    
    # Take the last iteration's thetas
    thetas = loop_summary["optimized_thetas"]
    
    println("Loaded circuit with $(length(circuit)) gates and $(length(thetas)) parameters")
    
    # Verify consistency
    if length(circuit) != length(thetas)
        @warn "Circuit length ($(length(circuit))) doesn't match thetas length ($(length(thetas)))!"
    end
    
    # Prepare Hamiltonian using type-based dispatch
    if backend == :reversediff
        # For ReverseDiff, pass functions that construct Hamiltonians
        hamiltonian_or_constructor = (CT, nq_val, topo) -> get_hamiltonian(system_hamiltonian, CT, nq_val, topo)
        topology_arg = topo_scaled
    else
        # For Mooncake/ForwardDiff, construct Hamiltonian directly
        hamiltonian_or_constructor = get_hamiltonian(system_hamiltonian, nq, topo_scaled)
        topology_arg = nothing
    end
    
    # Determine if this is an ED system for exact re-evaluation
    is_ed_system = nq <= max_exact_reeval
    
    # Re-evaluate using the MATCHING loss function based on backend and threads
    reevaluated_energy_per_qubit = 0.0
    
    try
        # Create a run dict with backend and threads info for run_lossfunction
        run_info = Dict("backend" => String(backend), "threads" => threads)
        
        reevaluated_energy = run_lossfunction(
            run_info, circuit, thetas, nq, hamiltonian_or_constructor, overlap_function;
            min_abs_coeff=reeval_min_abs_coeff, 
            max_weight=reeval_max_weight,
            max_freq=Inf,
            topology=topology_arg
        )
        reevaluated_energy_per_qubit = reevaluated_energy / nq
        println("✓ Re-evaluated using correct loss function")
    catch e
        @warn "Failed to re-evaluate with correct loss function" exception=e
        continue
    end
    
    # Store the result
    reevaluated_energies[nq] = reevaluated_energy_per_qubit
    
    # For small systems, also do exact re-evaluation
    if is_ed_system
        println("Small system (nq <= $max_exact_reeval) - performing exact re-evaluation (mw=Inf, mac=0.0)")
        try
            run_info = Dict("backend" => String(backend), "threads" => threads)
            
            exact_reevaluated_energy = run_lossfunction(
                run_info, circuit, thetas, nq, hamiltonian_or_constructor, overlap_function;
                min_abs_coeff=0.0, 
                max_weight=Inf,
                max_freq=Inf,
                topology=topology_arg
            )
            exact_reevaluated_energy_per_qubit = exact_reevaluated_energy / nq
            reevaluated_energies_exact[nq] = exact_reevaluated_energy_per_qubit
            println("✓ Exact re-evaluated energy/qubit: $exact_reevaluated_energy_per_qubit")
        catch e
            @warn "Failed exact re-evaluation for small system" exception=e
        end
    else
        println("Large system (nq > $max_exact_reeval) - using truncated re-evaluation only")
    end
    
    reeval_abs_error = reevaluated_energy_per_qubit - exact_energy
    reeval_rel_error = abs(reeval_abs_error / exact_energy)
    
    println("Re-evaluated energy/qubit: $reevaluated_energy_per_qubit")
    println("Reference GS energy/qubit ($method): $exact_energy")
    println("Re-evaluated abs error: $reeval_abs_error")
    println("Re-evaluated rel error: $reeval_rel_error")
end

println("\n" * "="^90)
println("Re-evaluation complete for $(length(reevaluated_energies)) systems")
println("="^90)

In [ ]:
# Prepare data for comprehensive relative error plot
# Configure which qubit sizes to ignore in the plot (e.g., [4, 20])
nq_ignore = []  # Set to empty array [] to include all qubit sizes

# Exclude specified qubit sizes to better show behavior for other systems
qubit_sizes_filtered = filter(x -> x ∉ nq_ignore, qubit_sizes_full)

title_text =  "Loop ADAPT-VQE Accuracy for $HAMILTONIAN_NAME"
println("Preparing data for plot: $title_text")

# Separate data by reference method (ED vs NQS)
ed_nqs_all = Int[]
nqs_nqs_all = Int[]

for (idx, nq) in enumerate(qubit_sizes_full)
    if nq ∈ nq_ignore
        continue  # Skip ignored qubit sizes
    end
    if haskey(reference_energies, nq)
        if reference_energies[nq]["method"] == "ED"
            push!(ed_nqs_all, nq)
        else
            push!(nqs_nqs_all, nq)
        end
    end
end
println("ED reference systems in plot: ", ed_nqs_all)
println("NQS reference systems in plot: ", nqs_nqs_all)

# Separate into above-GS and below-GS cases
above_gs_nq = Int[]
above_gs_errors = Float64[]
below_gs_nq = Int[]
below_gs_errors = Float64[]

for (idx, nq) in enumerate(qubit_sizes_full)
    if nq ∈ nq_ignore
        continue  # Skip ignored qubit sizes
    end
    if absolute_errors_full[idx] < 0
        # Below ground state (problematic)
        push!(below_gs_nq, nq)
        push!(below_gs_errors, relative_distances_full[idx])
    else
        # Above ground state (correct)
        push!(above_gs_nq, nq)
        push!(above_gs_errors, relative_distances_full[idx])
    end
end

# Format mac for display - with safety check
initial_mac_str = initial_mac !== nothing ? format_mac_latex(initial_mac) : "N/A"
initial_mw_str = initial_mw !== nothing ? string(initial_mw) : "N/A"

# Check which re-evaluated cases are still below GS
reeval_above_gs_nq = Int[]
reeval_above_gs_errors = Float64[]
reeval_below_gs_nq = Int[]
reeval_below_gs_errors = Float64[]
exact_reeval_above_nq = Int[]
exact_reeval_above_errors = Float64[]
exact_reeval_below_nq = Int[]
exact_reeval_below_errors = Float64[]

for (idx, nq) in enumerate(qubit_sizes_full)
    if nq ∈ nq_ignore
        continue  # Skip ignored qubit sizes
    end
    if haskey(reference_energies, nq)
        exact_energy = reference_energies[nq]["gs_per_site"]
        is_ed = reference_energies[nq]["method"] == "ED"
        
        # Check if we have exact re-evaluation for ED systems
        if is_ed && haskey(reevaluated_energies_exact, nq)
            # Use exact re-evaluation for ED systems
            exact_reeval_abs_error = reevaluated_energies_exact[nq] - exact_energy
            exact_reeval_rel_error = abs(exact_reeval_abs_error / exact_energy)
            
            if exact_reeval_abs_error < 0
                # Exact re-eval still below GS (should be very rare!)
                push!(exact_reeval_below_nq, nq)
                push!(exact_reeval_below_errors, exact_reeval_rel_error)
            else
                # Exact re-eval above GS (correct)
                push!(exact_reeval_above_nq, nq)
                push!(exact_reeval_above_errors, exact_reeval_rel_error)
            end
        else
            # Use standard re-evaluation for NQS or if exact failed
            if haskey(reevaluated_energies, nq)
                reeval_abs_error = reevaluated_energies[nq] - exact_energy
                reeval_rel_error = abs(reeval_abs_error / exact_energy)
                
                if reeval_abs_error < 0
                    # Still below GS even after re-evaluation (serious issue!)
                    push!(reeval_below_gs_nq, nq)
                    push!(reeval_below_gs_errors, reeval_rel_error)
                else
                    # Above GS after re-evaluation (correct)
                    push!(reeval_above_gs_nq, nq)
                    push!(reeval_above_gs_errors, reeval_rel_error)
                end
            end
        end
    end
end

# Format mac for display - get the LaTeX content without the wrapper
reeval_mac_latex = format_mac_latex(reeval_min_abs_coeff) 

println("Data preparation complete:")
println("  Above GS: $(length(above_gs_nq)) points")
println("  Below GS: $(length(below_gs_nq)) points")
println("  Exact reeval above: $(length(exact_reeval_above_nq)) points")
println("  Exact reeval below: $(length(exact_reeval_below_nq)) points")
println("  Approx reeval above: $(length(reeval_above_gs_nq)) points")
println("  Approx reeval below: $(length(reeval_below_gs_nq)) points")

In [ ]:
# Plot the comprehensive relative error data
println("Plot: $title_text")

# Choose plotting backend:
# Option 1: gr() - Fast, handles extreme values, but not native LaTeX rendering
# Option 2: pgfplotsx() - LaTeX/PDF quality, works well when ylims match yticks range
pgfplotsx()

p_comprehensive = plot(
    xlabel="Number of Qubits", 
    ylabel="Relative Energy Error",
    yscale=:log10,
    #yticks=([1e-12,1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1], 
    #       [L"10^{-12}", L"10^{-11}", L"10^{-10}", L"10^{-9}", L"10^{-8}", L"10^{-7}", L"10^{-6}", L"10^{-5}", L"10^{-4}", L"10^{-3}", L"10^{-2}", L"10^{-1}"]),
    #yticks = ( [1e-2, 1e-1], 
    #[L"10^{-2}", L"10^{-1}"]),
    minorticks=false,
    grid=true,
    #ylims = (5e-3, 2e-1),
    legend=(0.15,0.25),
    size=(750,500),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot above-GS cases
if !isempty(above_gs_nq)
    scatter!(p_comprehensive, above_gs_nq, above_gs_errors,
            marker=:circle,
            color=bold_hex[1],
            label=L"ADAPT above Reference GS (mac=%$initial_mac_str, mw=%$initial_mw_str)",
            markerstrokewidth=2)
end

# Plot below-GS cases 
if !isempty(below_gs_nq)
    scatter!(p_comprehensive, below_gs_nq, below_gs_errors,
            marker=:circle,
            #markersize=10,
            color=bold_hex[2],
            label=L"ADAPT below Reference GS (mac=%$initial_mac_str, mw=%$initial_mw_str)",
            markerstrokewidth=2)
end

# Plot exact re-evaluated errors above GS for ED systems (teal/cyan)
if !isempty(exact_reeval_above_nq)
    scatter!(p_comprehensive, exact_reeval_above_nq, exact_reeval_above_errors,
         marker=:square,
         #markersize=7,
         color=bold_hex[3],
         label="Exact re-eval above GS (mac=0, mw=Inf)",
         markerstrokewidth=1.5)
end

# Plot exact re-evaluated errors below GS for ED systems (magenta - should be rare!)
if !isempty(exact_reeval_below_nq)
    scatter!(p_comprehensive, exact_reeval_below_nq, exact_reeval_below_errors,
         marker=:square,
         #markersize=7,
         color=bold_hex[4],
         label="Exact re-eval below GS (mac=0, mw=Inf)",
         markerstrokewidth=1.5)
end

# Plot approx re-evaluated errors (above GS) in green
if !isempty(reeval_above_gs_nq)
    scatter!(p_comprehensive, reeval_above_gs_nq, reeval_above_gs_errors,
         marker=:square,
         #markersize=6,
         color=bold_hex[5],
         label = L"Approx re-eval above GS (mac=%$reeval_mac_latex, mw=%$reeval_max_weight)",
         markerstrokewidth=1.5)
end

# Plot approx re-evaluated errors (still below GS!) in orange
if !isempty(reeval_below_gs_nq)
    scatter!(p_comprehensive, reeval_below_gs_nq, reeval_below_gs_errors,
            marker=:square,
            #markersize=8,
            color=bold_hex[6],
            label=L"Approx re-eval below GS (mac=%$reeval_mac_latex, mw=%$reeval_max_weight)",
            markerstrokewidth=2)
end


# Check if plot has any data before displaying
total_data_points = length(above_gs_nq) + length(below_gs_nq) + 
                   length(exact_reeval_above_nq) + length(exact_reeval_below_nq) +
                   length(reeval_above_gs_nq) + length(reeval_below_gs_nq)

if total_data_points > 0
    display(p_comprehensive)
    #save_thesis_plot(p_comprehensive, title_text)
else
    @warn "No data to plot! All data arrays are empty. Check that:"
    println("  - Loop runs are loaded (length: $(length(hamiltonian_loop_runs)))")
    println("  - Reference energies exist (length: $(length(reference_energies)))")
    println("  - initial_mac = $initial_mac")
    println("  - initial_mw = $initial_mw")
end

We can inspect particular runs now that we know how accurate they are.

## Single Runs Evaluations (with Re-Evaluations)

### Configuration for Single Run Evaluation

Select a specific qubit count to analyze and set re-evaluation truncation parameters.

In [ ]:
# Select a specific qubit count to analyze
selected_nq = 80
# Re-evaluation truncation parameters
selected_reeval_mw = Inf
selected_reeval_mac = 1e-10

println("="^80)
println("SINGLE RUN EVALUATION CONFIGURATION")
println("="^80)
println("Selected qubit count: $selected_nq")
println("Re-evaluation max_weight: $selected_reeval_mw")
println("Re-evaluation min_abs_coeff: $selected_reeval_mac")
println("="^80)

### Load and Extract Iteration Data

In [ ]:
# Verify selected_nq exists in our data
if !haskey(original_data_by_nq, selected_nq)
    error("No data found for selected_nq=$selected_nq. Available: $(sort(collect(keys(original_data_by_nq))))")
end

# Load the original data file for selected_nq
original_data = original_data_by_nq[selected_nq]
all_results = original_data["all_results"]

# Use the last run (most accurate) to match our loop_summary
selected_run_idx = length(all_results)
selected_run = all_results[selected_run_idx]

println("\n" * "="^80)
println("ANALYZING RUN FOR $selected_nq QUBITS")
println("="^80)

# Get metadata
metadata = selected_run["sequence_metadata"]
println("MW sequence: $(metadata["mw_sequence"])")
println("MAC sequence: $(metadata["mac_sequence"])")
println("Min iters for stagnation: $(metadata["min_iters_for_stagnation_sequence"])")
println("Timestamp: $(metadata["timestamp"])")

# Get loop summary
loop_summary = selected_run["loop_summary"]
println("\nLoop Summary:")
println("  Total elapsed time: $(round(loop_summary["total_elapsed_time_s"], digits=2))s")
println("  Final energy/qubit: $(loop_summary["final_energy_per_qubit"])")
println("  Final circuit depth: $(loop_summary["final_circuit_depth"])")

# Extract reference energy
if haskey(reference_energies, selected_nq)
    reference_energy_per_site = reference_energies[selected_nq]["gs_per_site"]
    reference_method = reference_energies[selected_nq]["method"]
    println("\nReference GS energy/qubit ($reference_method): $reference_energy_per_site")
else
    @warn "No reference energy available for $selected_nq qubits"
    reference_energy_per_site = NaN
    reference_method = "N/A"
end

println("="^80)

In [ ]:
# Extract iteration data from all subruns
subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
num_subruns = length(subrun_keys)

println("\n" * "="^80)
println("EXTRACTING ITERATION DATA FROM $num_subruns SUBRUNS")
println("="^80)

# Initialize arrays for combined data
all_iteration_energies = Float64[]  # Energy at each iteration from run history
all_iteration_grads = Float64[]     # Max gradient at each iteration
all_iteration_thetas = Vector{Float64}[]  # Thetas at each iteration
iteration_boundaries = Int[]  # Where each subrun ends
subrun_info = []  # Info about each subrun (mw, mac)

cumulative_iterations = 0

for (subrun_idx, run_idx) in enumerate(subrun_keys)
    run_data = selected_run[run_idx]
    
    println("\nSubrun $subrun_idx (key=$run_idx):")
    println("  max_weight: $(run_data["max_weight"])")
    println("  min_abs_coeff: $(run_data["min_abs_coeff"])")
    println("  Iterations: $(run_data["num_iterations"])")
    println("  Converged: $(run_data["converged"])")
    
    # Store subrun info
    push!(subrun_info, (
        mw=run_data["max_weight"],
        mac=run_data["min_abs_coeff"],
        num_iters=run_data["num_iterations"]
    ))
    
    # Extract iteration data from this subrun
    thetas_per_iter = run_data["optimized_thetas_per_iter"]

    # Append to combined arrays
    append!(all_iteration_thetas, thetas_per_iter)
    
    # Update cumulative iterations and boundary
    cumulative_iterations += run_data["num_iterations"]
    push!(iteration_boundaries, cumulative_iterations)
end

# Extract combined data from loop_summary
all_iteration_energies = loop_summary["combined_energy_curve"]
all_iteration_grads = loop_summary["combined_max_grads"]

total_iterations = length(all_iteration_energies)
println("\n" * "="^80)
println("COMBINED DATA SUMMARY")
println("="^80)
println("Total iterations: $total_iterations")
println("Iteration boundaries: $iteration_boundaries")
println("Energy curve length: $(length(all_iteration_energies))")
println("Gradient curve length: $(length(all_iteration_grads))")
println("Thetas curve length: $(length(all_iteration_thetas))")
println("="^80)

### Re-evaluate Energies at Each Iteration

In [ ]:
# Re-evaluate energies at each iteration using the correct loss function
println("\n" * "="^80)
println("RE-EVALUATING ENERGIES AT EACH ITERATION")
println("="^80)
println("Using: max_weight=$selected_reeval_mw, min_abs_coeff=$selected_reeval_mac")

# Get backend info from first subrun
first_subrun = selected_run[subrun_keys[1]]
backend = Symbol(first_subrun["backend"])
threads = first_subrun["threads"]
println("Backend: $backend, Threads: $threads")

# Get topology
topo_scaled = get_topology_from_run(first_subrun)
#println(topo_scaled )
# Prepare Hamiltonian or constructor based on backend
if backend == :reversediff
    hamiltonian_or_constructor = hamiltonian_constructor_typed
    topology_arg = topo_scaled
else
    hamiltonian_or_constructor = hamiltonian_constructor_simple(selected_nq, topo_scaled)
    topology_arg = nothing
end

# Create run_info dict for run_lossfunction
run_info = Dict("backend" => String(backend), "threads" => threads)

# Re-evaluate at each iteration
reevaluated_iteration_energies = Float64[]
current_circuit = []  # Build circuit incrementally

println("\nRe-evaluating $(length(all_iteration_thetas)) iterations...")
for (iter, thetas) in enumerate(all_iteration_thetas)
    # Build circuit up to this iteration
    # Circuit grows by one gate per iteration
    if iter == 1
        # First iteration: get the first gate from loop_summary circuit
        current_circuit = [loop_summary["circuit"][1]]
    else
        # Add the next gate from the full circuit
        if iter <= length(loop_summary["circuit"])
            push!(current_circuit, loop_summary["circuit"][iter])
        else
            @warn "Iteration $iter exceeds circuit length $(length(loop_summary["circuit"]))"
            break
        end
    end
    
    # Re-evaluate with current circuit and thetas
    try
        reevaluated_energy = run_lossfunction(
            run_info, current_circuit, thetas, selected_nq, 
            hamiltonian_or_constructor, overlap_function;
            min_abs_coeff=selected_reeval_mac, 
            max_weight=selected_reeval_mw,
            max_freq=Inf,
            topology=topology_arg
        )
        reevaluated_energy_per_qubit = reevaluated_energy / selected_nq
        push!(reevaluated_iteration_energies, reevaluated_energy_per_qubit)
        
        # Print progress every 10 iterations
        if iter % 10 == 0 || iter == length(all_iteration_thetas)
            println("  Iteration $iter/$(length(all_iteration_thetas)): E/Q = $reevaluated_energy_per_qubit")
        end
    catch e
        @warn "Failed to re-evaluate iteration $iter" exception=e
        push!(reevaluated_iteration_energies, NaN)
    end
end

println("\n" * "="^80)
println("Re-evaluation complete: $(length(reevaluated_iteration_energies)) points")
println("="^80)

### Plot Energy Convergence with Re-evaluation

In [ ]:
title_text = "Energy Convergence for $selected_nq Qubits ($HAMILTONIAN_NAME)"
p_energy = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Energy per Qubit",
    legend=:topright,
    size=(725,500),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Get the mac and mw from the first subrun (initial parameters)
initial_mac = subrun_info[1].mac
initial_mw = subrun_info[1].mw

# Format mac for display
initial_mac_str = format_mac_latex(initial_mac)

# Plot original energies from run history
plot!(p_energy, 1:length(all_iteration_energies), all_iteration_energies,
      label=L"ADAPT (mac=%$initial_mac_str, mw=%$initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=4,
      color=bold_hex[1],
      alpha=1.0)

# Plot re-evaluated energies
if !isempty(reevaluated_iteration_energies)
    reeval_mac_str = format_mac_latex(selected_reeval_mac)
    plot!(p_energy, 1:length(reevaluated_iteration_energies), reevaluated_iteration_energies,
          label=L"Re-evaluated (mac=%$reeval_mac_str, mw=%$selected_reeval_mw)",
          linewidth=2,
          marker=:square,
          markersize=4,
          color=bold_hex[2],
          linestyle=:dash,
          alpha=1.0)
end

# Add ground state reference line
if !isnan(reference_energy_per_site)
    hline!(p_energy, [reference_energy_per_site],
           linestyle=:dash,
           linewidth=2.5,
           color=:grey,
           label="Reference GS")#($reference_method)")
end

# Add vertical lines at subrun boundaries with MW and MAC labels
for (idx, boundary) in enumerate(iteration_boundaries[1:end-1])
    mw_current = subrun_info[idx].mw
    mac_current = subrun_info[idx].mac
    mw_next = subrun_info[idx+1].mw
    mac_next = subrun_info[idx+1].mac
    
    # Format MAC values for display
    mac_current_str = format_mac_latex(mac_current)
    mac_next_str = format_mac_latex(mac_next)
    
    # Create label showing both MW and MAC transitions
    transition_label = L"MW %$mw_current→%$mw_next, MAC %$mac_current_str→%$mac_next_str"
    
    vline!(p_energy, [boundary], 
           linestyle=:dot, 
           linewidth=2,
           color=:purple,
           label=transition_label,
           alpha=0.6)
end

# Add text annotations for each subrun region
# for (idx, info) in enumerate(subrun_info)
#     # Calculate midpoint of this subrun
#     start_iter = idx == 1 ? 1 : iteration_boundaries[idx-1] + 1
#     end_iter = iteration_boundaries[idx]
#     mid_iter = (start_iter + end_iter) / 2
    
#     # Get y-position (top of plot)
#     y_pos = maximum(all_iteration_energies) * 0.98
    
#     # Add annotation
#     annotate!(p_energy, mid_iter, y_pos, 
#               text("MW=$(info.mw)\nMAC=$(info.mac)", 8, :center, :gray))
# end

display(p_energy)
#save_thesis_plot(p_energy, title_text)

### Log-Log plot of Energy Convergence

In [ ]:
title_text = "Log Log Energy Convergence for $selected_nq Qubits ($HAMILTONIAN_NAME)"
p_energy_log = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Absolute Energy per Qubit",
    legend=:topleft,
    xaxis=:log,
    yaxis=:log,
    size=(725,500),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Get the mac and mw from the first subrun (initial parameters)
initial_mac = subrun_info[1].mac
initial_mw = subrun_info[1].mw

# Format mac for display
initial_mac_str = format_mac_latex(initial_mac)

# Plot original energies from run history
plot!(p_energy_log, 1:length(all_iteration_energies), abs.(all_iteration_energies),
      label=L"ADAPT (mac=%$initial_mac_str, mw=%$initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=4,
      color=bold_hex[1],
      alpha=1.0)

# Plot re-evaluated energies
if !isempty(reevaluated_iteration_energies)
    reeval_mac_str = format_mac_latex(selected_reeval_mac)
    plot!(p_energy_log, 1:length(reevaluated_iteration_energies), abs.(reevaluated_iteration_energies),
          label=L"Re-evaluated (mac=%$reeval_mac_str, mw=%$selected_reeval_mw)",
          linewidth=2,
          marker=:square,
          markersize=4,
          color=bold_hex[2],
          linestyle=:dash,
          alpha=1.0)
end

# Add ground state reference line
if !isnan(reference_energy_per_site)
    hline!(p_energy_log, [-reference_energy_per_site],
           linestyle=:dash,
           linewidth=2.5,
           color=:grey,
           label="Reference GS ($reference_method)")
end
display(p_energy_log)
#save_thesis_plot(p_energy_log, title_text)

In [ ]:
title_text = "Relative Error of Energy Convergence for $selected_nq Qubits ($HAMILTONIAN_NAME)"
p_energy_log = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Absolute Energy Error",
    legend=:bottomleft,
    #xaxis=:log,
    yaxis=:log,
    xticks = ([1,10,20,40],
        ["1", "10", "20", "40"]),
    yticks = ([1e-8,1e-6,1e-4, 1e-2],
    [L"10^{-8}", L"10^{-6}", L"10^{-4}",L"10^{-2}"]),
    size=(725,500),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Get the mac and mw from the first subrun (initial parameters)
initial_mac = subrun_info[1].mac
initial_mw = subrun_info[1].mw

# Format mac for display
initial_mac_str = format_mac_latex(initial_mac)

# Plot original energies from run history
plot!(p_energy_log, 1:length(all_iteration_energies), abs.(all_iteration_energies.-reference_energy_per_site),
      label=L"ADAPT (mac=%$initial_mac_str, mw=%$initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=4,
      color=bold_hex[1],
      alpha=1.0)

# Plot re-evaluated energies
if !isempty(reevaluated_iteration_energies)
    reeval_mac_str = format_mac_latex(selected_reeval_mac)
    plot!(p_energy_log, 1:length(reevaluated_iteration_energies), abs.(reevaluated_iteration_energies.-reference_energy_per_site),
          label=L"Re-evaluated (mac=%$reeval_mac_str, mw=%$selected_reeval_mw)",
          linewidth=2,
          marker=:square,
          markersize=4,
          color=bold_hex[2],
          linestyle=:dash,
          alpha=1.0)
end
display(p_energy_log)
#save_thesis_plot(p_energy_log, title_text)

### Plot Gradient Convergence

In [ ]:
title_text ="Gradient Convergence for $selected_nq Qubits ($HAMILTONIAN_NAME)"
p_grads = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Max Absolute Gradient",
    legend=:bottomleft,
    size=(725,500),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Plot gradient magnitudes
plot!(p_grads, 1:length(all_iteration_grads), abs.(all_iteration_grads),
      label="Max Absolute Gradient",
      linewidth=2,
      marker=:circle,
      markersize=4,
      color=bold_hex[1])

# Add vertical lines at subrun boundaries with MW and MAC labels
for (idx, boundary) in enumerate(iteration_boundaries[1:end-1])
    mw_current = subrun_info[idx].mw
    mac_current = subrun_info[idx].mac
    mw_next = subrun_info[idx+1].mw
    mac_next = subrun_info[idx+1].mac
    
    # Format MAC values for display
    mac_current_str = format_mac_latex(mac_current)
    mac_next_str = format_mac_latex(mac_next)
    
    # Create label showing both MW and MAC transitions
    transition_label = L"MW %$mw_current→%$mw_next, MAC %$mac_current_str→%$mac_next_str"
    
    # vline!(p_grads, [boundary], 
    #        linestyle=:dot, 
    #        linewidth=2,
    #        color=:purple,
    #        label=transition_label,
    #        alpha=0.6)
end

# Add text annotations for each subrun region
for (idx, info) in enumerate(subrun_info)
    # Calculate midpoint of this subrun
    start_iter = idx == 1 ? 1 : iteration_boundaries[idx-1] + 1
    end_iter = iteration_boundaries[idx]
    mid_iter = (start_iter + end_iter) / 2
    
    # Get y-position (top of plot in log scale)
    y_pos = maximum(abs.(all_iteration_grads)) * 0.5
    
    # Format MAC for annotation
    mac_str = format_mac_latex(info.mac)

end

display(p_grads)
#save_thesis_plot(p_grads, title_text)

### All Truncation Parameters Runs Comparison

Compare all truncation parameter runs with their re-evaluations. Cluster runs are shown in bold colors, while re-evaluations are shown in pastel/transparent colors.

In [ ]:
# Configuration: Select which runs to include in the comparison
# Toggle which runs you want to plot (true = include, false = exclude)
# Run indices: 1 = first (least accurate), end = last (most accurate)

# Get available runs for selected_nq
if haskey(original_data_by_nq, selected_nq)
    original_data = original_data_by_nq[selected_nq]
    all_results = original_data["all_results"]
    num_available_runs = length(all_results)
    
    println("="^80)
    println("AVAILABLE RUNS FOR nq=$selected_nq")
    println("="^80)
    println("Total available runs: $num_available_runs")
    
    # Print information about each run
    for run_idx in 1:num_available_runs
        run_data = all_results[run_idx]
        metadata = run_data["sequence_metadata"]
        println("\nRun $run_idx:")
        println("  MW sequence: $(metadata["mw_sequence"])")
        println("  MAC sequence: $(metadata["mac_sequence"])")
    end
    
    # By default, include all runs
    # You can modify this array to select specific runs, e.g., [1, 3, 5] or [2:4...]
    runs_to_include = collect(1:num_available_runs)  # Include all runs
    # Alternative examples:
    #runs_to_include = [num_available_runs-1, num_available_runs]  # Only last 2 runs
    # runs_to_include = [1, num_available_runs]  # Only first and last
    #runs_to_include = [1,4]
    println("\n" * "="^80)
    println("SELECTED RUNS TO PLOT: $runs_to_include")
    println("="^80)
else
    error("No data found for selected_nq=$selected_nq")
end

In [ ]:
# Extract iteration data for all selected runs
println("\n" * "="^80)
println("EXTRACTING DATA FOR ALL SELECTED RUNS")
println("="^80)

# Store data for each run
all_runs_data = []

for run_idx in runs_to_include
    run_data = all_results[run_idx]
    metadata = run_data["sequence_metadata"]
    
    println("\n" * "-"^80)
    println("Processing Run $run_idx")
    println("-"^80)
    println("MW sequence: $(metadata["mw_sequence"])")
    println("MAC sequence: $(metadata["mac_sequence"])")
    
    # Extract iteration data from all subruns
    subrun_keys = sort([k for k in keys(run_data) if k isa Integer])
    
    # Initialize arrays for this run
    run_iteration_energies = Float64[]
    run_iteration_thetas = Vector{Float64}[]
    run_subrun_info = []
    run_iteration_boundaries = Int[]
    
    cumulative_iterations = 0
    
    for (subrun_idx, subrun_key) in enumerate(subrun_keys)
        subrun = run_data[subrun_key]
        
        # Store subrun info
        push!(run_subrun_info, (
            mw=subrun["max_weight"],
            mac=subrun["min_abs_coeff"],
            num_iters=subrun["num_iterations"]
        ))
        
        # Extract thetas
        thetas_per_iter = subrun["optimized_thetas_per_iter"]
        append!(run_iteration_thetas, thetas_per_iter)
        
        # Update cumulative iterations
        cumulative_iterations += subrun["num_iterations"]
        push!(run_iteration_boundaries, cumulative_iterations)
    end
    
    # Get energy curve from loop_summary
    loop_summary = run_data["loop_summary"]
    run_iteration_energies = loop_summary["combined_energy_curve"]
    
    println("  Total iterations: $(length(run_iteration_energies))")
    println("  Final energy/qubit: $(loop_summary["final_energy_per_qubit"])")
    
    # Store all data for this run
    push!(all_runs_data, (
        run_idx=run_idx,
        metadata=metadata,
        energies=run_iteration_energies,
        thetas=run_iteration_thetas,
        subrun_info=run_subrun_info,
        boundaries=run_iteration_boundaries,
        loop_summary=loop_summary,
        circuit=loop_summary["circuit"]
    ))
end

println("\n" * "="^80)
println("Data extraction complete for $(length(all_runs_data)) runs")
println("="^80)

In [ ]:
# Re-evaluate all selected runs with the same truncation parameters
println("\n" * "="^80)
println("RE-EVALUATING ALL SELECTED RUNS")
println("="^80)
println("Using: max_weight=$selected_reeval_mw, min_abs_coeff=$selected_reeval_mac")

# Get backend info from first subrun
first_subrun = selected_run[subrun_keys[1]]
backend = Symbol(first_subrun["backend"])
threads = first_subrun["threads"]
println("Backend: $backend, Threads: $threads")

# Get topology
topo_scaled = get_topology_from_run(first_subrun)

# Prepare Hamiltonian or constructor
if backend == :reversediff
    hamiltonian_or_constructor = hamiltonian_constructor_typed
    topology_arg = topo_scaled
else
    hamiltonian_or_constructor = hamiltonian_constructor_simple(selected_nq, topo_scaled)
    topology_arg = nothing
end

run_info = Dict("backend" => String(backend), "threads" => threads)

# Re-evaluate each run
for run_data in all_runs_data
    println("\n" * "-"^80)
    println("Re-evaluating Run $(run_data.run_idx)")
    println("-"^80)
    
    reevaluated_energies = Float64[]
    current_circuit = []
    
    for (iter, thetas) in enumerate(run_data.thetas)
        # Build circuit incrementally
        if iter == 1
            current_circuit = [run_data.circuit[1]]
        else
            if iter <= length(run_data.circuit)
                push!(current_circuit, run_data.circuit[iter])
            else
                @warn "Iteration $iter exceeds circuit length $(length(run_data.circuit))"
                break
            end
        end
        
        # Re-evaluate
        try
            reevaluated_energy = run_lossfunction(
                run_info, current_circuit, thetas, selected_nq, 
                hamiltonian_or_constructor, overlap_function;
                min_abs_coeff=selected_reeval_mac, 
                max_weight=selected_reeval_mw,
                max_freq=Inf,
                topology=topology_arg
            )
            push!(reevaluated_energies, reevaluated_energy / selected_nq)
            
            # Print progress
            if iter % 20 == 0 || iter == length(run_data.thetas)
                println("  Iteration $iter/$(length(run_data.thetas))")
            end
        catch e
            @warn "Failed to re-evaluate iteration $iter of run $(run_data.run_idx)" exception=e
            push!(reevaluated_energies, NaN)
        end
    end
    
    # Store re-evaluated energies in the run_data (mutate the tuple as a NamedTuple)
    # Since tuples are immutable, we'll create a new entry in all_runs_data
    # Actually, we can't mutate, so we'll keep them separate
    run_data_dict = Dict(
        :run_idx => run_data.run_idx,
        :reevaluated_energies => reevaluated_energies
    )
    
    # Store in a new array
    if !@isdefined(all_runs_reevaluated)
        global all_runs_reevaluated = []
    end
    push!(all_runs_reevaluated, run_data_dict)
    
    println("  Re-evaluation complete: $(length(reevaluated_energies)) points")
end

println("\n" * "="^80)
println("Re-evaluation complete for all runs")
println("="^80)

In [ ]:
# Plot all runs with cluster runs in bold and re-evaluations in pastel
title_text = "All Truncation Parameter Runs Comparison for $HAMILTONIAN_NAME"
println("Plot: $title_text")

p_all_runs = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Energy per Qubit",
    legend=:topright,
    size=(900, 600),
    left_margin=10Plots.mm,
    bottom_margin=8Plots.mm
)

# Plot each run
for (idx, run_data) in enumerate(all_runs_data)
    run_idx = run_data.run_idx
    
    # Create label with truncation parameters
    mw_seq = run_data.metadata["mw_sequence"]
    mac_seq = run_data.metadata["mac_sequence"]
    mac_seq_latex = format_mac_vector_latex(mac_seq; vector_brackets=false)
    #label_base = L"mac=%$mac_seq_latex, mw=%$(mw_seq[1]), " # TODO hardcoded for single truncations!!
    label_base = latexstring("mac=\$$(mac_seq_latex)\$, mw=\$$(mw_seq[1])\$")
    # Get color for this run from the bold palette
    color_idx = mod1(idx, length(bold))
    bold_color = bold[color_idx]
    pastel_color = pastels[color_idx]
    
    # Plot cluster energies (BOLD)
    plot!(p_all_runs, 1:length(run_data.energies), run_data.energies,
          label=label_base,
          marker=:circle,
          markersize=5,
          color=bold_color,
          alpha=1.0)
    
    # Plot re-evaluated energies (PASTEL)
    if @isdefined(all_runs_reevaluated)
        # Find corresponding re-evaluated data
        reeval_data = findfirst(x -> x[:run_idx] == run_idx, all_runs_reevaluated)
        if !isnothing(reeval_data)
            reevaluated_energies = all_runs_reevaluated[reeval_data][:reevaluated_energies]
            if !isempty(reevaluated_energies)

                selected_reeval_mac_latex = format_mac_latex(selected_reeval_mac)
                plot!(p_all_runs, 1:length(reevaluated_energies), reevaluated_energies,
                      label=L"Re-eval (mac=%$selected_reeval_mac_latex, mw=%$selected_reeval_mw)",
                      color=pastel_color,
                      linestyle=:dash,
                      marker=:square,
                      markersize=4,
                      alpha=0.8)
            end
        end
    end
end

# Add ground state reference line
if !isnan(reference_energy_per_site)
    hline!(p_all_runs, [reference_energy_per_site],
           linestyle=:dash,
           linewidth=2.5,
           color=:grey,
           label="Reference GS")# ($reference_method)")
end

display(p_all_runs)
#save_thesis_plot(p_all_runs,title_text)

In [ ]:
# Plot absolute energy error in log scale comparing all runs
title_text = "Absolute Energy Error Comparison for $selected_nq Qubits ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_energy_log = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Absolute Energy Error",
    legend=:bottomright,
    yaxis=:log,
    yticks = ([1e-8,1e-6,1e-4, 1e-2],
        [L"10^{-8}", L"10^{-6}", L"10^{-4}",L"10^{-2}"]),
    size=(900,650),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Plot each run's absolute error
for (idx, run_data) in enumerate(all_runs_data)
    run_idx = run_data.run_idx
    
    # Get truncation parameters
    initial_mac = run_data.subrun_info[1].mac
    initial_mw = run_data.subrun_info[1].mw
    initial_mac_str = format_mac_latex(initial_mac)
    
    # Get color for this run
    color_idx = mod1(idx, length(bold_hex))
    bold_color = bold_hex[color_idx]
    pastel_color = pastels[color_idx]
    
    # Calculate absolute error for original energies
    if !isnan(reference_energy_per_site)
        abs_errors = abs.(run_data.energies .- reference_energy_per_site)
        
        # Plot original energies (BOLD)
        plot!(p_energy_log, 1:length(abs_errors), abs_errors,
              label=L"mac=%$initial_mac_str, mw=%$initial_mw",
              linewidth=2,
              marker=:circle,
              markersize=4,
              color=bold_color,
              alpha=1.0)
        
        # Plot re-evaluated energies (PASTEL)
        if @isdefined(all_runs_reevaluated)
            reeval_data = findfirst(x -> x[:run_idx] == run_idx, all_runs_reevaluated)
            if !isnothing(reeval_data)
                reevaluated_energies = all_runs_reevaluated[reeval_data][:reevaluated_energies]
                if !isempty(reevaluated_energies)
                    reeval_abs_errors = abs.(reevaluated_energies .- reference_energy_per_site)
                    
                    selected_reeval_mac_latex = format_mac_latex(selected_reeval_mac)
                    plot!(p_energy_log, 1:length(reeval_abs_errors), reeval_abs_errors,
                          label=L"Re-eval (mac=%$selected_reeval_mac_latex, mw=%$selected_reeval_mw)",
                          linewidth=2,
                          marker=:square,
                          markersize=4,
                          color=pastel_color,
                          linestyle=:dash,
                          alpha=0.8)
                end
            end
        end
    else
        @warn "Reference energy not available, cannot plot absolute errors"
    end
end

display(p_energy_log)
#save_thesis_plot(p_energy_log, title_text)

### Timing and Memory of Single Run

In [ ]:
# Extract total timing and memory for all runs
println("\n" * "="^80)
println("TIMING AND MEMORY SUMMARY FOR ALL SELECTED RUNS")
println("="^80)

for run_data in all_runs_data
    run_idx = run_data.run_idx
    loop_summary = run_data.loop_summary
    
    println("\n" * "-"^80)
    println("Run $run_idx")
    println("-"^80)
    
    # Get total timing from loop_summary
    if haskey(loop_summary, "total_elapsed_time_s")
        total_time_s = loop_summary["total_elapsed_time_s"]
        println("  Total elapsed time: $(round(total_time_s, digits=2))s ($(round(total_time_s/60, digits=2)) min)")
    else
        println("  Total elapsed time: N/A")
    end
    
    # Get total memory from loop_summary if available
    if haskey(loop_summary, "total_memory_gb")
        total_memory_gb = loop_summary["total_memory_gb"]
        println("  Total memory: $(round(total_memory_gb, digits=2)) GB")
    elseif haskey(loop_summary, "total_memory_bytes")
        total_memory_bytes = loop_summary["total_memory_bytes"]
        total_memory_gb = total_memory_bytes / 1e9
        println("  Total memory: $(round(total_memory_gb, digits=2)) GB")
    else
        println("  Total memory: N/A")
    end
    
    # Get peak RSS memory if available
    if haskey(loop_summary, "peak_rss_mb")
        peak_rss_mb = loop_summary["peak_rss_mb"]
        peak_rss_gb = peak_rss_mb / 1024.0
        println("  Peak RSS memory: $(round(peak_rss_mb, digits=2)) MB ($(round(peak_rss_gb, digits=2)) GB)")
    elseif haskey(loop_summary, "peak_rss_bytes")
        peak_rss_bytes = loop_summary["peak_rss_bytes"]
        peak_rss_mb = peak_rss_bytes / 1e6
        peak_rss_gb = peak_rss_bytes / 1e9
        println("  Peak RSS memory: $(round(peak_rss_mb, digits=2)) MB ($(round(peak_rss_gb, digits=2)) GB)")
    else
        println("  Peak RSS memory: N/A")
    end
    
    # Get per-subrun timing if available
    println("\n  Subrun breakdown:")
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    for (subrun_idx, subrun_key) in enumerate(subrun_keys)
        if subrun_key <= length(run_data.subrun_info)
            subrun = selected_run[subrun_key]
            subrun_info = run_data.subrun_info[subrun_idx]
            
            # Display truncation parameters
            mac_str = format_mac_latex(subrun_info.mac)
            print("    Subrun $subrun_idx (mw=$(subrun_info.mw), mac=$mac_str): ")
            
            # Try to get timing for this subrun
            if haskey(subrun, "elapsed_time_s")
                subrun_time = subrun["elapsed_time_s"]
                print("$(round(subrun_time, digits=2))s")
            else
                print("timing N/A")
            end
            
            # Try to get memory for this subrun
            if haskey(subrun, "memory_gb")
                subrun_mem = subrun["memory_gb"]
                print(", $(round(subrun_mem, digits=2)) GB")
            elseif haskey(subrun, "memory_bytes")
                subrun_mem_gb = subrun["memory_bytes"] / 1e9
                print(", $(round(subrun_mem_gb, digits=2)) GB")
            else
                print(", memory N/A")
            end
            
            # Try to get peak RSS memory for this subrun
            if haskey(subrun, "peak_rss_mb")
                subrun_peak_rss_mb = subrun["peak_rss_mb"]
                print(", peak RSS: $(round(subrun_peak_rss_mb, digits=2)) MB")
            elseif haskey(subrun, "peak_rss_bytes")
                subrun_peak_rss_mb = subrun["peak_rss_bytes"] / 1e6
                print(", peak RSS: $(round(subrun_peak_rss_mb, digits=2)) MB")
            end
            println()
        end
    end
end

println("\n" * "="^80)
println("Timing and memory summary complete")
println("="^80)

In [ ]:
# Extract gradient data for all selected runs
println("\n" * "="^80)
println("EXTRACTING GRADIENT DATA FOR ALL SELECTED RUNS")
println("="^80)

# Store gradient data for each run
all_runs_gradients = []

for run_data in all_runs_data
    run_idx = run_data.run_idx
    
    # Get combined gradient curve from loop_summary
    loop_summary = run_data.loop_summary
    
    if haskey(loop_summary, "combined_max_grads")
        run_gradients = loop_summary["combined_max_grads"]
        
        push!(all_runs_gradients, (
            run_idx=run_idx,
            gradients=run_gradients
        ))
        
        println("Run $run_idx: $(length(run_gradients)) gradient values")
    else
        @warn "No gradient data found for run $run_idx"
    end
end

println("\n" * "="^80)
println("Gradient extraction complete for $(length(all_runs_gradients)) runs")
println("="^80)

In [ ]:
# Plot gradient convergence comparison for all runs
title_text = "Gradient Convergence Comparison for $selected_nq Qubits ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_all_runs_grads = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Max Absolute Gradient",
    legend=:topright,
    size=(900, 600),
    left_margin=10Plots.mm,
    bottom_margin=8Plots.mm
)

# Plot each run's gradients
for (idx, run_data) in enumerate(all_runs_data)
    # Get color for this run (same as energy plot) from bold palette
    color_idx = mod1(idx, length(bold))
    bold_color = bold_hex[color_idx]
    
    run_idx = run_data.run_idx
    
    # Find gradient data for this run
    grad_data_idx = findfirst(x -> x.run_idx == run_idx, all_runs_gradients)
    
    if !isnothing(grad_data_idx)
        gradients = all_runs_gradients[grad_data_idx].gradients
        
        # Create label with proper string interpolation
        mw_seq = run_data.metadata["mw_sequence"]
        mac_seq = run_data.metadata["mac_sequence"]
        
        # Format MAC sequence as LaTeX
        mac_latex = format_mac_vector_latex(mac_seq; vector_brackets=false)
        
        # Build label using regular string interpolation, then wrap in latexstring
        label = latexstring("mac=\$$(mac_latex)\$, mw=\$$(mw_seq[1])\$")
        
        # Plot gradient magnitudes
        plot!(p_all_runs_grads, 1:length(gradients), abs.(gradients),
              label=label, marker = :circle, markersize=5,
              color=bold_color)
    end
end

display(p_all_runs_grads)
#save_thesis_plot(p_all_runs_grads, title_text)
println("\n" * "="^80)
println("GRADIENT PLOT SUMMARY")
println("="^80)
println("Number of runs plotted: $(length(all_runs_gradients))")
println("Y-axis: Log scale for gradient magnitudes")
println("="^80)

### V-score Computation for the single Run

In [ ]:
# V-score Computation for Single Run
vscore = false  # Set to true to enable computation
if vscore
    # Get the selected run data
    loop_summary = hamiltonian_loop_runs[selected_nq]
    
    # Get circuit and optimized parameters
    circuit = loop_summary["circuit"]
    thetas = loop_summary["optimized_thetas"]
    
    # Get topology for the system
    if haskey(original_data_by_nq, selected_nq)
        original_data = original_data_by_nq[selected_nq]
        all_results = original_data["all_results"]
        run_keys = sort(collect(keys(all_results)))
        selected_run = all_results[run_keys[end]]
        subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
        first_subrun = selected_run[subrun_keys[1]]
        topo_scaled = get_topology_from_run(first_subrun)
    else
        error("No topology data found for nq=$selected_nq")
    end
    
    # Construct Hamiltonian
    hamiltonian = hamiltonian_constructor_simple(selected_nq, topo_scaled)
    
    # Get reference energy for E_inf (ground state energy)
    if haskey(reference_energies, selected_nq)
        E_inf = reference_energies[selected_nq]["ground_state"]  # Total energy, not per site
    else
        @warn "No reference energy found for nq=$selected_nq, using 0.0"
        E_inf = 0.0
    end
    
    # Compute V-score with re-evaluation truncation parameters
    vscore_single = vscore_calculation_MC(
        thetas, 
        circuit, 
        selected_nq, 
        hamiltonian, 
        overlap_function; verbose=true,
        min_abs_coeff_vscore=selected_reeval_mac,  # From re-evaluation config: 1e-10
        max_weight_vscore=selected_reeval_mw,      # From re-evaluation config: Inf
        E_inf=E_inf
    )
    vscore = 1.43e-6*90/(-134.99999978681228)^2
    println("hand calc:",vscore)
    println("\n" * "="^80)
    println("V-SCORE CALCULATION FOR SINGLE RUN")
    println("="^80)
    println("System size: $selected_nq qubits")
    println("Truncation: max_weight=$selected_reeval_mw, min_abs_coeff=$selected_reeval_mac")
    println("Reference energy E_inf: $E_inf")
    println("V-score: $vscore_single")
    println("="^80)
end

### Compare Re-Runs of the Single Run

In [ ]:
# Analyze rerun data from the selected nq
println("\n" * "="^80)
println("RERUN ANALYSIS FOR nq=$selected_nq")
println("="^80)

# Get the loop summary for the selected nq
if haskey(hamiltonian_loop_runs, selected_nq)
    loop_summary = hamiltonian_loop_runs[selected_nq]
    
    # Backward compatibility: check if rerun fields exist
    if haskey(loop_summary, "num_reruns")
        # Extract rerun information
        num_reruns = loop_summary["num_reruns"]
        best_rerun_idx = loop_summary["best_rerun_idx"]
        all_rerun_final_energies = loop_summary["all_rerun_final_energies"]
        
        println("Number of reruns: $num_reruns")
        println("Best rerun index: $best_rerun_idx")
        println("\nFinal energies from all reruns:")
        println(all_rerun_final_energies)
        
        if num_reruns > 1
            # Compare timing and memory metrics
            println("\n" * "-"^80)
            println("TIMING AND MEMORY COMPARISON")
            println("-"^80)
            
            # Extract timing and memory data
            total_elapsed_time = loop_summary["total_elapsed_time_s"]
            best_rerun_elapsed_time = loop_summary["best_rerun_elapsed_time_s"]
            all_rerun_elapsed_times = loop_summary["all_rerun_elapsed_times"]
            
            total_gc_time = loop_summary["total_gc_time_s"]
            best_rerun_gc_time = loop_summary["best_rerun_gc_time_s"]
            all_rerun_gc_times = loop_summary["all_rerun_gc_times"]
            
            total_memory = loop_summary["total_memory_bytes"]
            best_rerun_memory = loop_summary["best_rerun_memory_bytes"]
            all_rerun_memory_bytes = loop_summary["all_rerun_memory_bytes"]
            
            # Calculate averages
            avg_elapsed_time = total_elapsed_time / num_reruns
            avg_gc_time = total_gc_time / num_reruns
            avg_memory = total_memory / num_reruns
            
            # Calculate statistics for energies
            using Statistics
            avg_energy = mean(all_rerun_final_energies)
            std_energy = std(all_rerun_final_energies)
            min_energy = minimum(all_rerun_final_energies)
            max_energy = maximum(all_rerun_final_energies)
            
            println("\n Energy Statistics:")
            println("  Best (selected):  $(all_rerun_final_energies[best_rerun_idx]) (rerun $best_rerun_idx)")
            println("  Average:          $avg_energy")
            println("  Std deviation:    $std_energy")
            println("  Min:              $min_energy")
            println("  Max:              $max_energy")
            println("  Range:            $(max_energy - min_energy)")
            
            println("\n Timing Statistics:")
            println("  Total elapsed (all reruns): $(round(total_elapsed_time, digits=2))s")
            println("  Average per rerun:          $(round(avg_elapsed_time, digits=2))s")
            println("  Best run time:              $(round(best_rerun_elapsed_time, digits=2))s")
            println("  Best vs average:            $(round(best_rerun_elapsed_time/avg_elapsed_time*100, digits=1))%")
            
            println("\n GC Time Statistics:")
            println("  Total GC (all reruns): $(round(total_gc_time, digits=2))s")
            println("  Average per rerun:     $(round(avg_gc_time, digits=2))s")
            println("  Best run GC:           $(round(best_rerun_gc_time, digits=2))s")
            println("  Best vs average:       $(round(best_rerun_gc_time/avg_gc_time*100, digits=1))%")
            
            println("\n Memory Statistics:")
            println("  Total memory (all reruns): $(round(total_memory/1e9, digits=3)) GB")
            println("  Average per rerun:         $(round(avg_memory/1e9, digits=3)) GB")
            println("  Best run memory:           $(round(best_rerun_memory/1e9, digits=3)) GB")
            println("  Best vs average:           $(round(best_rerun_memory/avg_memory*100, digits=1))%")
            
            # Check if best run had best or worst timing
            best_time_idx = argmin(all_rerun_elapsed_times)
            worst_time_idx = argmax(all_rerun_elapsed_times)
            
            println("\n Performance Summary:")
            if best_rerun_idx == best_time_idx
                println("  ✓ Best energy run was also the FASTEST (rerun $best_time_idx)")
            elseif best_rerun_idx == worst_time_idx
                println("  ⚠ Best energy run was the SLOWEST (rerun $worst_time_idx)")
            else
                println("  • Best energy: rerun $best_rerun_idx")
                println("  • Fastest:     rerun $best_time_idx ($(round(all_rerun_elapsed_times[best_time_idx], digits=2))s)")
                println("  • Slowest:     rerun $worst_time_idx ($(round(all_rerun_elapsed_times[worst_time_idx], digits=2))s)")
            end
        else
            println("\nOnly 1 rerun performed - no comparison available")
        end
    else
        # Old run format without rerun data
        println(" This is an old run format (no rerun data available)")
        println("Final energy/qubit: $(loop_summary["final_energy_per_qubit"])")
        println("Circuit depth: $(loop_summary["final_circuit_depth"])")
        println("\nTo use rerun analysis, re-run the benchmark with num_reruns > 1")
    end
else
    @warn "No data found for nq=$selected_nq in hamiltonian_loop_runs"
end

println("\n" * "="^80)

## ADAPT-Methods comparison
### Accuracy

In [ ]:
# Load comparison method data across all qubit sizes
println("\n" * "="^90)
println("LOADING COMPARISON ADAPT METHODS")
println("="^90)

# Dictionary to store all methods: method_name => {nq => loop_summary}
all_methods_data = Dict{String, Dict{Int, Dict{String, Any}}}()

# Load main benchmark directory
all_methods_data[method_legend_main] = hamiltonian_loop_runs

# Try to load comparison directory 1
if isdir(benchmark_dir_compare_1)
    println("\nLoading comparison method 1: $benchmark_dir_compare_1")
    try
        compare_runs_1 = load_loop_master_logs_all_nq(
            benchmark_dir_compare_1; 
            hamiltonian_name=String(HAMILTONIAN_NAME),
            tile_nq=tile_nq_compare_1,
            min_nq=min_nq,
            max_nq=max_nq,
            use_most_accurate=true
        )
        all_methods_data[method_legend_compare_1] = compare_runs_1
    catch e
        @warn "Failed to load comparison method 1" exception=e
    end
else
    @warn "Comparison directory 1 not found: $benchmark_dir_compare_1"
end

# Try to load comparison directory 2
if @isdefined(benchmark_dir_compare_2) && isdir(benchmark_dir_compare_2)
    println("\nLoading comparison method 2: $benchmark_dir_compare_2")
    try
        compare_runs_2 = load_loop_master_logs_all_nq(
            benchmark_dir_compare_2; 
            hamiltonian_name=String(HAMILTONIAN_NAME),
            tile_nq=tile_nq_compare_2,
            min_nq=min_nq,
            max_nq=max_nq,
            use_most_accurate=true
        )
        all_methods_data[method_legend_compare_2] = compare_runs_2
    catch e
        @warn "Failed to load comparison method 2" exception=e
    end
elseif @isdefined(benchmark_dir_compare_2)
    @warn "Comparison directory 2 not found: $benchmark_dir_compare_2"
else
    println("\nNo comparison directory 2 defined (benchmark_dir_compare_2 not set)")
end

println("\n" * "="^90)
println("LOADED $(length(all_methods_data)) ADAPT METHOD(S)")
println("="^90)
for (method_name, method_data) in all_methods_data
    nqs_available = sort(collect(keys(method_data)))
    println("$method_name: $(length(nqs_available)) system sizes - $nqs_available")
end
println("="^90)

In [ ]:
# Check what files are actually in the benchmark directory
println("Files in benchmark_dir ($benchmark_dir):")
if isdir(benchmark_dir)
    all_files = readdir(benchmark_dir)
    jld2_files = filter(f -> endswith(f, ".jld2"), all_files)
    println("Total .jld2 files: ", length(jld2_files))
    println("\nLoop master logs:")
    for f in filter(f -> startswith(f, "master_log_loop"), jld2_files)
        println("  - $f")
    end
else
    println("Directory not found!")
end

In [ ]:
# Load original data for all methods to access backend info
println("\n" * "="^90)
println("LOADING ORIGINAL DATA FILES FOR ALL METHODS")
println("="^90)

all_methods_original_data = Dict{String, Dict{Int, Any}}()

# Function to load original data for a directory with NEW naming format
function load_original_data_for_dir(dir_path, hamiltonian_name, topology_name, tile_nq, nq_range)
    original_data_dict = Dict{Int, Any}()
    
    all_files = filter(f -> endswith(f, ".jld2"), readdir(dir_path))
    
    # NEW pattern: master_log_loop_SystemHamiltonian(:Name)_SystemTopology(:topology)_...
    master_logs = filter(all_files) do f
        startswith(f, "master_log_loop_SystemHamiltonian(:$(hamiltonian_name))_SystemTopology(:$(topology_name))")
    end
    
    println("  Searching for pattern: master_log_loop_SystemHamiltonian(:$(hamiltonian_name))_SystemTopology(:$(topology_name))")
    println("  Found $(length(master_logs)) matching files")
    
    for filename in master_logs
        if occursin(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
            m = match(r"_tile(\d+)_scaled(\d+)\.jld2$", filename)
            file_tile_nq = parse(Int, m.captures[1])
            file_scaled_nq = parse(Int, m.captures[2])
            
            if file_tile_nq == tile_nq && file_scaled_nq in nq_range
                filepath = joinpath(dir_path, filename)
                try
                    data = JLD2.load(filepath)
                    if haskey(data, "all_results")
                        original_data_dict[file_scaled_nq] = data
                        println("    ✓ Loaded nq=$file_scaled_nq from $filename")
                    end
                catch e
                    @warn "Failed to load $filename" exception=e
                end
            end
        end
    end
    
    return original_data_dict
end

# Load for main method
println("Loading original data for $method_legend_main...")
if haskey(all_methods_data, method_legend_main)
    nq_range = collect(keys(all_methods_data[method_legend_main]))
    all_methods_original_data[method_legend_main] = load_original_data_for_dir(
        benchmark_dir, String(HAMILTONIAN_NAME), String(TOPOLOGY_NAME), tile_nq, nq_range
    )
    println("✓ Loaded $(length(all_methods_original_data[method_legend_main])) files for $method_legend_main")
end

# Load for comparison method 1
if haskey(all_methods_data, method_legend_compare_1)
    println("\nLoading original data for $method_legend_compare_1...")
    nq_range = collect(keys(all_methods_data[method_legend_compare_1]))
    all_methods_original_data[method_legend_compare_1] = load_original_data_for_dir(
        benchmark_dir_compare_1, String(HAMILTONIAN_NAME), String(TOPOLOGY_NAME), tile_nq_compare_1, nq_range
    )
    println("✓ Loaded $(length(all_methods_original_data[method_legend_compare_1])) files for $method_legend_compare_1")
end

# Load for comparison method 2
if haskey(all_methods_data, method_legend_compare_2)
    println("\nLoading original data for $method_legend_compare_2...")
    nq_range = collect(keys(all_methods_data[method_legend_compare_2]))
    all_methods_original_data[method_legend_compare_2] = load_original_data_for_dir(
        benchmark_dir_compare_2, String(HAMILTONIAN_NAME), String(TOPOLOGY_NAME), tile_nq_compare_2, nq_range
    )
    println("✓ Loaded $(length(all_methods_original_data[method_legend_compare_2])) files for $method_legend_compare_2")
end

println("\n" * "="^90)
println("Original data loaded for $(length(all_methods_original_data)) method(s)")
println("="^90)

In [ ]:
# Re-evaluate all methods across all qubit sizes for accuracy comparison
println("\n" * "="^90)
println("RE-EVALUATING ALL METHODS FOR ACCURACY COMPARISON")
println("="^90)
println("Re-evaluation: max_weight=$reeval_max_weight, min_abs_coeff=$reeval_min_abs_coeff")

# Dictionary to store re-evaluated energies: method_name => {nq => energy_per_qubit}
all_methods_reevaluated = Dict{String, Dict{Int, Float64}}()

# Define explicit order of methods to maintain consistent ordering (main, compare1, compare2)
method_order = [method_legend_main]
if haskey(all_methods_data, method_legend_compare_1)
    push!(method_order, method_legend_compare_1)
end
if haskey(all_methods_data, method_legend_compare_2)
    push!(method_order, method_legend_compare_2)
end
println(method_order)

for method_name in method_order
    method_data = all_methods_data[method_name]
    println("\n" * "-"^90)
    println("Method: $method_name")
    println("-"^90)
    
    reevaluated_energies_method = Dict{Int, Float64}()
    
    # Get original data for this method
    if !haskey(all_methods_original_data, method_name)
        @warn "No original data found for $method_name, skipping"
        continue
    end
    
    original_data_method = all_methods_original_data[method_name]
    
    for nq in sort(collect(keys(method_data)))
        loop_summary = method_data[nq]
        
        # Get backend info from original data
        if !haskey(original_data_method, nq)
            @warn "No original data for $method_name nq=$nq, skipping"
            continue
        end
        
        original_data = original_data_method[nq]
        all_results = original_data["all_results"]
        
        # all_results is an array, use length to get the last (most accurate) run
        selected_run_idx = length(all_results)
        selected_run = all_results[selected_run_idx]
        
        # Get backend from first subrun
        subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
        if isempty(subrun_keys)
            @warn "No subruns for $method_name nq=$nq"
            continue
        end
        
        first_subrun = selected_run[subrun_keys[1]]
        backend = Symbol(first_subrun["backend"])
        threads = first_subrun["threads"]
        
        # Get topology
        topo_scaled = get_topology_from_run(first_subrun)
        
        # Prepare Hamiltonian or constructor
        if backend == :reversediff
            hamiltonian_or_constructor = hamiltonian_constructor_typed
            topology_arg = topo_scaled
        else
            hamiltonian_or_constructor = hamiltonian_constructor_simple(nq, topo_scaled)
            topology_arg = nothing
        end
        
        # Get circuit and thetas
        circuit = loop_summary["circuit"]
        thetas = loop_summary["optimized_thetas"]
        
        # Re-evaluate
        try
            run_info = Dict("backend" => String(backend), "threads" => threads)
            reevaluated_energy = run_lossfunction(
                run_info, circuit, thetas, nq, 
                hamiltonian_or_constructor, overlap_function;
                min_abs_coeff=reeval_min_abs_coeff, 
                max_weight=reeval_max_weight,
                max_freq=Inf,
                topology=topology_arg
            )
            reevaluated_energy_per_qubit = reevaluated_energy / nq
            reevaluated_energies_method[nq] = reevaluated_energy_per_qubit
            
            println("  nq=$nq: Re-evaluated E/Q = $reevaluated_energy_per_qubit")
        catch e
            @warn "Failed to re-evaluate $method_name nq=$nq" exception=e
        end
    end
    
    all_methods_reevaluated[method_name] = reevaluated_energies_method
    println("✓ Re-evaluated $(length(reevaluated_energies_method)) systems for $method_name")
end

println("\n" * "="^90)
println("Re-evaluation complete for $(length(all_methods_reevaluated)) method(s)")
println("="^90)

In [ ]:
# Plot accuracy comparison: Re-evaluated relative errors for all methods
println("\n" * "="^90)
println("PLOTTING ACCURACY COMPARISON")
println("="^90)

pgfplotsx()
# Collect all relative errors to determine y-axis range
all_rel_errors = Float64[]

for method_name in method_order
    if !haskey(all_methods_reevaluated, method_name)
        continue
    end
    reevaluated_energies = all_methods_reevaluated[method_name]
    for nq in sort(collect(keys(reevaluated_energies)))
        if haskey(reference_energies, nq)
            ref_energy = reference_energies[nq]["gs_per_site"]
            reeval_energy = reevaluated_energies[nq]
            abs_error = reeval_energy - ref_energy
            rel_error = abs(abs_error / ref_energy)
            # Set floor at 10^-15 to avoid numerical issues
            rel_error = max(rel_error, 1e-15)
            push!(all_rel_errors, rel_error)
        end
    end
end

# Determine appropriate y-axis range with padding
min_error = minimum(all_rel_errors)
max_error = maximum(all_rel_errors)

# Add padding: go one order of magnitude wider on each side
log_min = floor(log10(min_error))
log_max = ceil(log10(max_error))
y_min = 10^(log_min - 1)  # One order of magnitude below
y_max = 10^(log_max + 1)  # One order of magnitude above

println("Data range: $(min_error) to $(max_error)")
println("Plot range: $(y_min) to $(y_max)")

title_text = "ADAPT Methods Accuracy Comparison ($HAMILTONIAN_NAME)"
p_accuracy = plot(
    xlabel="Number of Qubits", 
    ylabel="Relative Energy Error",
    yscale=:log10,
    ylims=(y_min, y_max),
    yticks=([1e-12,1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1], 
            [L"10^{-12}", L"10^{-11}", L"10^{-10}", L"10^{-9}", L"10^{-8}", L"10^{-7}", L"10^{-6}", L"10^{-5}", L"10^{-4}", L"10^{-3}", L"10^{-2}", L"10^{-1}"]),
    minorticks=false,
    grid=true,
    legend=:right,
    size=(900, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot each method

mac_latex = format_mac_latex(reeval_min_abs_coeff)
for (method_idx, method_name) in enumerate(method_order)
    if !haskey(all_methods_reevaluated, method_name)
        continue
    end
    reevaluated_energies = all_methods_reevaluated[method_name]
    nqs = Int[]
    rel_errors = Float64[]
    
    for nq in sort(collect(keys(reevaluated_energies)))
        if haskey(reference_energies, nq)
            ref_energy = reference_energies[nq]["gs_per_site"]
            reeval_energy = reevaluated_energies[nq]
            
            abs_error = reeval_energy - ref_energy
            rel_error = abs(abs_error / ref_energy)
            # Set floor at 10^-15 to avoid numerical issues
            rel_error = max(rel_error, 1e-15)
            
            push!(nqs, nq)
            push!(rel_errors, rel_error)
        end
    end
    
    if !isempty(nqs)
        scatter!(p_accuracy, nqs, rel_errors,
                marker=:circle,
                markersize=7,
                color=bold_hex[method_idx],
                label=L"%$(method_name) (Re-eval mac=%$mac_latex, mw=%$reeval_max_weight)",
                markerstrokewidth=2)
        
        # Connect with lines
        plot!(p_accuracy, nqs, rel_errors,
              linewidth=2,
              color=bold_hex[method_idx],
              label="",
              alpha=0.5)
    end
end

# Add ED/NQS boundary
# ed_nqs = [nq for nq in keys(reference_energies) if reference_energies[nq]["method"] == "ED"]
# nqs_nqs = [nq for nq in keys(reference_energies) if reference_energies[nq]["method"] == "NQS"]

# if !isempty(ed_nqs) && !isempty(nqs_nqs)
#     max_ed_nq = maximum(ed_nqs)
#     vline!(p_accuracy, [max_ed_nq + 0.5], 
#           linestyle=:dot, 
#           linewidth=2, 
#           color=:gray, 
#           label="ED/NQS boundary",
#           alpha=0.5)
# end

display(p_accuracy)
#save_thesis_plot(p_accuracy, title_text)

### Single Run Comparison of Methods

Compare specific runs from the main method and comparison method, showing energy convergence, gradient convergence, and re-evaluations for both methods side-by-side.

In [ ]:
# Configuration for comparing specific runs from each method
comparison_nq = 45  # Number of qubits to compare

# For main method: which subrun index (from all_results array)
main_method_run_idx = length(all_methods_original_data[method_legend_main][comparison_nq]["all_results"])  # Use last (most accurate) run

# For comparison method: which subrun index
comparison_method_run_idx = main_method_run_idx # Use first run (or adjust as needed)

# Re-evaluation parameters for comparison
comparison_reeval_mw = Inf
comparison_reeval_mac = 1e-10

println("="^80)
println("SINGLE RUN COMPARISON CONFIGURATION")
println("="^80)
println("Qubit count: $comparison_nq")
println("Main method run index: $main_method_run_idx")
println("Comparison method run index: $comparison_method_run_idx")
println("Re-evaluation max_weight: $comparison_reeval_mw")
println("Re-evaluation min_abs_coeff: $comparison_reeval_mac")
println("="^80)

In [ ]:
# Load data for MAIN METHOD
println("\n" * "="^80)
println("LOADING MAIN METHOD DATA")
println("="^80)

# Verify comparison_nq exists
if !haskey(all_methods_original_data[method_legend_main], comparison_nq)
    error("No data found for comparison_nq=$comparison_nq in main method. Available: $(sort(collect(keys(all_methods_original_data[method_legend_main]))))")
end

# Load original data
main_original_data = all_methods_original_data[method_legend_main][comparison_nq]
main_all_results = main_original_data["all_results"]

if main_method_run_idx > length(main_all_results) || main_method_run_idx < 1
    error("Invalid main_method_run_idx=$main_method_run_idx. Available: 1 to $(length(main_all_results))")
end

main_selected_run = main_all_results[main_method_run_idx]
main_metadata = main_selected_run["sequence_metadata"]

println("Main method metadata:")
println("  MW sequence: $(main_metadata["mw_sequence"])")
println("  MAC sequence: $(main_metadata["mac_sequence"])")
println("  Timestamp: $(main_metadata["timestamp"])")

# Extract main method subrun data
main_subrun_keys = sort([k for k in keys(main_selected_run) if k isa Integer])
main_num_subruns = length(main_subrun_keys)

main_all_iteration_energies = Float64[]
main_all_iteration_grads = Float64[]
main_all_iteration_thetas = Vector{Float64}[]
main_iteration_boundaries = Int[]
main_subrun_info = []

main_cumulative_iterations = 0

for (subrun_idx, run_idx) in enumerate(main_subrun_keys)
    run_data = main_selected_run[run_idx]
    
    push!(main_subrun_info, (
        mw=run_data["max_weight"],
        mac=run_data["min_abs_coeff"],
        num_iters=run_data["num_iterations"]
    ))
    
    thetas_per_iter = run_data["optimized_thetas_per_iter"]
    append!(main_all_iteration_thetas, thetas_per_iter)
    
    main_cumulative_iterations += run_data["num_iterations"]
    push!(main_iteration_boundaries, main_cumulative_iterations)
end

# Extract combined data from loop_summary
main_loop_summary = main_selected_run["loop_summary"]
main_all_iteration_energies = main_loop_summary["combined_energy_curve"]
main_all_iteration_grads = main_loop_summary["combined_max_grads"]

println("  Total subruns: $main_num_subruns")
println("  Total iterations: $(length(main_all_iteration_energies))")
println("  Iteration boundaries: $main_iteration_boundaries")
println("="^80)

In [ ]:
# Load data for COMPARISON METHOD
println("\n" * "="^80)
println("LOADING COMPARISON METHOD DATA")
println("="^80)

# Verify comparison method exists and has data for comparison_nq
if !haskey(all_methods_original_data, method_legend_compare_1)
    error("Comparison method not loaded: $method_legend_compare_1")
end

if !haskey(all_methods_original_data[method_legend_compare_1], comparison_nq)
    error("No comparison data found for nq=$comparison_nq. Available: $(sort(collect(keys(all_methods_original_data[method_legend_compare_1]))))")
end

# Load original data
comp_original_data = all_methods_original_data[method_legend_compare_1][comparison_nq]
comp_all_results = comp_original_data["all_results"]

if comparison_method_run_idx > length(comp_all_results) || comparison_method_run_idx < 1
    error("Invalid comparison_method_run_idx=$comparison_method_run_idx. Available: 1 to $(length(comp_all_results))")
end

comp_selected_run = comp_all_results[comparison_method_run_idx]
comp_metadata = comp_selected_run["sequence_metadata"]

println("Comparison method metadata:")
println("  MW sequence: $(comp_metadata["mw_sequence"])")
println("  MAC sequence: $(comp_metadata["mac_sequence"])")
println("  Timestamp: $(comp_metadata["timestamp"])")

# Extract comparison method subrun data
comp_subrun_keys = sort([k for k in keys(comp_selected_run) if k isa Integer])
comp_num_subruns = length(comp_subrun_keys)

comp_all_iteration_energies = Float64[]
comp_all_iteration_grads = Float64[]
comp_all_iteration_thetas = Vector{Float64}[]
comp_iteration_boundaries = Int[]
comp_subrun_info = []

comp_cumulative_iterations = 0

for (subrun_idx, run_idx) in enumerate(comp_subrun_keys)
    run_data = comp_selected_run[run_idx]
    
    push!(comp_subrun_info, (
        mw=run_data["max_weight"],
        mac=run_data["min_abs_coeff"],
        num_iters=run_data["num_iterations"]
    ))
    
    thetas_per_iter = run_data["optimized_thetas_per_iter"]
    append!(comp_all_iteration_thetas, thetas_per_iter)
    
    comp_cumulative_iterations += run_data["num_iterations"]
    push!(comp_iteration_boundaries, comp_cumulative_iterations)
end

# Extract combined data from loop_summary
comp_loop_summary = comp_selected_run["loop_summary"]
comp_all_iteration_energies = comp_loop_summary["combined_energy_curve"]
comp_all_iteration_grads = comp_loop_summary["combined_max_grads"]

println("  Total subruns: $comp_num_subruns")
println("  Total iterations: $(length(comp_all_iteration_energies))")
println("  Iteration boundaries: $comp_iteration_boundaries")
println("="^80)

In [ ]:
# Re-evaluate energies for MAIN METHOD
println("\n" * "="^80)
println("RE-EVALUATING MAIN METHOD ENERGIES")
println("="^80)
println("Using: max_weight=$comparison_reeval_mw, min_abs_coeff=$comparison_reeval_mac")

# Get backend info from first subrun
main_first_subrun = main_selected_run[main_subrun_keys[1]]
main_backend = Symbol(main_first_subrun["backend"])
main_threads = main_first_subrun["threads"]
println("Backend: $main_backend, Threads: $main_threads")

# Get topology
main_topo_scaled = get_topology_from_run(main_first_subrun)

# Prepare Hamiltonian or constructor
if main_backend == :reversediff
    main_hamiltonian_or_constructor = hamiltonian_constructor_typed
    main_topology_arg = main_topo_scaled
else
    main_hamiltonian_or_constructor = hamiltonian_constructor_simple(comparison_nq, main_topo_scaled)
    main_topology_arg = nothing
end

# Create run_info dict
main_run_info = Dict("backend" => String(main_backend), "threads" => main_threads)

# Re-evaluate at each iteration
main_reevaluated_iteration_energies = Float64[]
main_current_circuit = []

println("Re-evaluating $(length(main_all_iteration_thetas)) iterations...")
for (iter, thetas) in enumerate(main_all_iteration_thetas)
    if iter == 1
        main_current_circuit = [main_loop_summary["circuit"][1]]
    else
        if iter <= length(main_loop_summary["circuit"])
            push!(main_current_circuit, main_loop_summary["circuit"][iter])
        else
            @warn "Iteration $iter exceeds circuit length $(length(main_loop_summary["circuit"]))"
            break
        end
    end
    
    try
        reevaluated_energy = run_lossfunction(
            main_run_info, main_current_circuit, thetas, comparison_nq, 
            main_hamiltonian_or_constructor, overlap_function;
            min_abs_coeff=comparison_reeval_mac, 
            max_weight=comparison_reeval_mw,
            max_freq=Inf,
            topology=main_topology_arg
        )
        reevaluated_energy_per_qubit = reevaluated_energy / comparison_nq
        push!(main_reevaluated_iteration_energies, reevaluated_energy_per_qubit)
        
        if iter % 10 == 0 || iter == length(main_all_iteration_thetas)
            println("  Iteration $iter/$(length(main_all_iteration_thetas)): E/Q = $reevaluated_energy_per_qubit")
        end
    catch e
        @warn "Failed to re-evaluate iteration $iter" exception=e
        push!(main_reevaluated_iteration_energies, NaN)
    end
end

println("Re-evaluation complete: $(length(main_reevaluated_iteration_energies)) points")
println("="^80)

In [ ]:
# Re-evaluate energies for COMPARISON METHOD
println("\n" * "="^80)
println("RE-EVALUATING COMPARISON METHOD ENERGIES")
println("="^80)
println("Using: max_weight=$comparison_reeval_mw, min_abs_coeff=$comparison_reeval_mac")

# Get backend info from first subrun
comp_first_subrun = comp_selected_run[comp_subrun_keys[1]]
comp_backend = Symbol(comp_first_subrun["backend"])
comp_threads = comp_first_subrun["threads"]
println("Backend: $comp_backend, Threads: $comp_threads")

# Get topology
comp_topo_scaled = get_topology_from_run(comp_first_subrun)

# Prepare Hamiltonian or constructor
if comp_backend == :reversediff
    comp_hamiltonian_or_constructor = hamiltonian_constructor_typed
    comp_topology_arg = comp_topo_scaled
else
    comp_hamiltonian_or_constructor = hamiltonian_constructor_simple(comparison_nq, comp_topo_scaled)
    comp_topology_arg = nothing
end

# Create run_info dict
comp_run_info = Dict("backend" => String(comp_backend), "threads" => comp_threads)

# Re-evaluate at each iteration
comp_reevaluated_iteration_energies = Float64[]
comp_current_circuit = []

println("Re-evaluating $(length(comp_all_iteration_thetas)) iterations...")
for (iter, thetas) in enumerate(comp_all_iteration_thetas)
    if iter == 1
        comp_current_circuit = [comp_loop_summary["circuit"][1]]
    else
        if iter <= length(comp_loop_summary["circuit"])
            push!(comp_current_circuit, comp_loop_summary["circuit"][iter])
        else
            @warn "Iteration $iter exceeds circuit length $(length(comp_loop_summary["circuit"]))"
            break
        end
    end
    
    try
        reevaluated_energy = run_lossfunction(
            comp_run_info, comp_current_circuit, thetas, comparison_nq, 
            comp_hamiltonian_or_constructor, overlap_function;
            min_abs_coeff=comparison_reeval_mac, 
            max_weight=comparison_reeval_mw,
            max_freq=Inf,
            topology=comp_topology_arg
        )
        reevaluated_energy_per_qubit = reevaluated_energy / comparison_nq
        push!(comp_reevaluated_iteration_energies, reevaluated_energy_per_qubit)
        
        if iter % 10 == 0 || iter == length(comp_all_iteration_thetas)
            println("  Iteration $iter/$(length(comp_all_iteration_thetas)): E/Q = $reevaluated_energy_per_qubit")
        end
    catch e
        @warn "Failed to re-evaluate iteration $iter" exception=e
        push!(comp_reevaluated_iteration_energies, NaN)
    end
end

println("Re-evaluation complete: $(length(comp_reevaluated_iteration_energies)) points")
println("="^80)

In [ ]:
# Plot Energy Convergence Comparison
println("\n" * "="^80)
println("PLOTTING ENERGY CONVERGENCE COMPARISON")
println("="^80)

title_text = "Energy Convergence Comparison for $comparison_nq Qubits ($HAMILTONIAN_NAME)"
p_energy_comparison = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Energy per Qubit",
    legend=:topright,
    size=(900, 600),
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Get initial parameters for main method
main_initial_mac = main_subrun_info[1].mac
main_initial_mw = main_subrun_info[1].mw
main_initial_mac_str = format_mac_latex(main_initial_mac)

# Get initial parameters for comparison method
comp_initial_mac = comp_subrun_info[1].mac
comp_initial_mw = comp_subrun_info[1].mw
comp_initial_mac_str = format_mac_latex(comp_initial_mac)

# Plot main method original energies
plot!(p_energy_comparison, 1:length(main_all_iteration_energies), main_all_iteration_energies,
      label=L"%$(method_legend_main) (init. mac=%$main_initial_mac_str, mw=%$main_initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=5,
      color=bold_hex[1],
      alpha=1.0)

# Plot main method re-evaluated energies
if !isempty(main_reevaluated_iteration_energies)
    main_reeval_mac_str = format_mac_latex(comparison_reeval_mac)
    plot!(p_energy_comparison, 1:length(main_reevaluated_iteration_energies), main_reevaluated_iteration_energies,
          label=L"%$(method_legend_main) re-eval (mac=%$main_reeval_mac_str, mw=%$comparison_reeval_mw)",
          linewidth=2,
          marker=:circle,
          markersize=5,
          color=pastels[1],
          alpha=1.0)
end

# Add vertical lines for main method subrun boundaries (if looped)
if main_num_subruns > 1
    for (idx, boundary) in enumerate(main_iteration_boundaries[1:end-1])
        mw_current = main_subrun_info[idx].mw
        mac_current = main_subrun_info[idx].mac
        mw_next = main_subrun_info[idx+1].mw
        mac_next = main_subrun_info[idx+1].mac
        
        mac_current_str = format_mac_latex(mac_current)
        mac_next_str = format_mac_latex(mac_next)
        
        transition_label = L"%$(method_legend_main): mw %$mw_current→%$mw_next, mac %$mac_current_str→%$mac_next_str"
        
        vline!(p_energy_comparison, [boundary], 
               linestyle=:dot, 
               linewidth=2,
               color=bold_hex[1],
               label=transition_label,
               alpha=0.5)
    end
end
# Plot comparison method original energies
plot!(p_energy_comparison, 1:length(comp_all_iteration_energies), comp_all_iteration_energies,
      label=L"%$(method_legend_compare_1) (mac=%$comp_initial_mac_str, mw=%$comp_initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=5,
      color=bold_hex[2],
      alpha=1.0)

# Plot comparison method re-evaluated energies
if !isempty(comp_reevaluated_iteration_energies)
    comp_reeval_mac_str = format_mac_latex(comparison_reeval_mac)
    plot!(p_energy_comparison, 1:length(comp_reevaluated_iteration_energies), comp_reevaluated_iteration_energies,
          label=L"%$(method_legend_compare_1) re-eval (mac=%$comp_reeval_mac_str, mw=%$comparison_reeval_mw)",
          linewidth=2,
          marker=:circle,
          markersize=5,
          color=pastels[2],
          alpha=1.0)
end

# Add ground state reference line
if haskey(reference_energies, comparison_nq)
    reference_energy_per_site = reference_energies[comparison_nq]["gs_per_site"]
    reference_method = reference_energies[comparison_nq]["method"]
    hline!(p_energy_comparison, [reference_energy_per_site],
           linestyle=:dash,
           linewidth=2.5,
           color=:grey,
           label="Reference GS")# ($reference_method)")
end

# Add vertical lines for comparison method subrun boundaries (if looped)
if comp_num_subruns > 1
    for (idx, boundary) in enumerate(comp_iteration_boundaries[1:end-1])
        mw_current = comp_subrun_info[idx].mw
        mac_current = comp_subrun_info[idx].mac
        mw_next = comp_subrun_info[idx+1].mw
        mac_next = comp_subrun_info[idx+1].mac
        
        mac_current_str = format_mac_latex(mac_current)
        mac_next_str = format_mac_latex(mac_next)
        
        transition_label = L"%$(method_legend_compare_1): mw %$mw_current→%$mw_next, mac %$mac_current_str→%$mac_next_str"
        
        vline!(p_energy_comparison, [boundary], 
               linestyle=:dashdot, 
               linewidth=2,
               color=bold_hex[3],
               label=transition_label,
               alpha=0.5)
    end
end

display(p_energy_comparison)
#save_thesis_plot(p_energy_comparison, title_text)

In [ ]:
# Plot Absolute Error Convergence Comparison
if haskey(reference_energies, comparison_nq)
    reference_energy_per_site = reference_energies[comparison_nq]["gs_per_site"]
    
    println("\n" * "="^80)
    println("PLOTTING ERROR CONVERGENCE COMPARISON")
    println("="^80)

    title_text_error = "Energy Error Convergence Comparison for $comparison_nq Qubits ($HAMILTONIAN_NAME)"
    p_error_comparison = plot(
        xlabel="ADAPT Iteration", 
        ylabel="Absolute Error",
        legend=:topright,
        size=(900, 600),
        left_margin=20Plots.mm,
        right_margin=10Plots.mm,
        top_margin=10Plots.mm,
        bottom_margin=10Plots.mm,
        yaxis=:log10
    )

    # Plot main method original errors
    plot!(p_error_comparison, 1:length(main_all_iteration_energies), abs.(main_all_iteration_energies .- reference_energy_per_site),
          label=L"%$(method_legend_main) (init. mac=%$main_initial_mac_str, mw=%$main_initial_mw)",
          linewidth=2,
          marker=:circle,
          markersize=5,
          color=bold_hex[1],
          alpha=1.0)

    # Plot main method re-evaluated errors
    if !isempty(main_reevaluated_iteration_energies)
        main_reeval_mac_str = format_mac_latex(comparison_reeval_mac)
        plot!(p_error_comparison, 1:length(main_reevaluated_iteration_energies), abs.(main_reevaluated_iteration_energies .- reference_energy_per_site),
              label=L"%$(method_legend_main) re-eval (mac=%$main_reeval_mac_str, mw=%$comparison_reeval_mw)",
              linewidth=2,
              marker=:circle,
              markersize=5,
              color=pastels[1],
              alpha=1.0)
    end

    # Add vertical lines for main method subrun boundaries (if looped)
    if main_num_subruns > 1
        for (idx, boundary) in enumerate(main_iteration_boundaries[1:end-1])
            mw_current = main_subrun_info[idx].mw
            mac_current = main_subrun_info[idx].mac
            mw_next = main_subrun_info[idx+1].mw
            mac_next = main_subrun_info[idx+1].mac
            
            mac_current_str = format_mac_latex(mac_current)
            mac_next_str = format_mac_latex(mac_next)
            
            transition_label = L"%$(method_legend_main): mw %$mw_current→%$mw_next, mac %$mac_current_str→%$mac_next_str"
            
            vline!(p_error_comparison, [boundary], 
                   linestyle=:dot, 
                   linewidth=2,
                   color=bold_hex[1],
                   label=transition_label,
                   alpha=0.5)
        end
    end

    # Plot comparison method original errors
    plot!(p_error_comparison, 1:length(comp_all_iteration_energies), abs.(comp_all_iteration_energies .- reference_energy_per_site),
          label=L"%$(method_legend_compare_1) (mac=%$comp_initial_mac_str, mw=%$comp_initial_mw)",
          linewidth=2,
          marker=:circle,
          markersize=5,
          color=bold_hex[2],
          alpha=1.0)

    # Plot comparison method re-evaluated errors
    if !isempty(comp_reevaluated_iteration_energies)
        comp_reeval_mac_str = format_mac_latex(comparison_reeval_mac)
        plot!(p_error_comparison, 1:length(comp_reevaluated_iteration_energies), abs.(comp_reevaluated_iteration_energies .- reference_energy_per_site),
              label=L"%$(method_legend_compare_1) re-eval (mac=%$comp_reeval_mac_str, mw=%$comparison_reeval_mw)",
              linewidth=2,
              marker=:circle,
              markersize=5,
              color=pastels[2],
              alpha=1.0)
    end

    # Add vertical lines for comparison method subrun boundaries (if looped)
    if comp_num_subruns > 1
        for (idx, boundary) in enumerate(comp_iteration_boundaries[1:end-1])
            mw_current = comp_subrun_info[idx].mw
            mac_current = comp_subrun_info[idx].mac
            mw_next = comp_subrun_info[idx+1].mw
            mac_next = comp_subrun_info[idx+1].mac
            
            mac_current_str = format_mac_latex(mac_current)
            mac_next_str = format_mac_latex(mac_next)
            
            transition_label = L"%$(method_legend_compare_1): mw %$mw_current→%$mw_next, mac %$mac_current_str→%$mac_next_str"
            
            vline!(p_error_comparison, [boundary], 
                   linestyle=:dashdot, 
                   linewidth=2,
                   color=bold_hex[3],
                   label=transition_label,
                   alpha=0.5)
        end
    end

    display(p_error_comparison)
    #save_thesis_plot(p_error_comparison, title_text_error)
else
    println("Reference energy not available for this qubit count ($comparison_nq). Cannot plot absolute error.")
end

In [ ]:
# Plot Gradient Convergence Comparison
println("\n" * "="^80)
println("PLOTTING GRADIENT CONVERGENCE COMPARISON")
println("="^80)

title_text_grad = "Gradient Convergence Comparison for $comparison_nq Qubits ($HAMILTONIAN_NAME)"
p_gradient_comparison = plot(
    xlabel="ADAPT Iteration", 
    ylabel="Maximum Gradient",
    legend=:topright,
    size=(900, 600),
    yscale=:log10,
    left_margin=20Plots.mm,
    right_margin=10Plots.mm,
    top_margin=10Plots.mm,
    bottom_margin=10Plots.mm
)

# Plot main method gradients
plot!(p_gradient_comparison, 1:length(main_all_iteration_grads), main_all_iteration_grads,
      label=L"%$(method_legend_main) (mac=%$main_initial_mac_str, mw=%$main_initial_mw)",
      linewidth=2,
      marker=:circle,
      markersize=3,
      color=bold_hex[1],
      alpha=0.8)

# Plot comparison method gradients
plot!(p_gradient_comparison, 1:length(comp_all_iteration_grads), comp_all_iteration_grads,
      label=L"%$(method_legend_compare_1) (mac=%$comp_initial_mac_str, mw=%$comp_initial_mw)",
      linewidth=2,
      marker=:diamond,
      markersize=3,
      color=bold_hex[3],
      alpha=0.8)

# Add vertical lines for main method subrun boundaries (if looped)
if main_num_subruns > 1
    for (idx, boundary) in enumerate(main_iteration_boundaries[1:end-1])
        mw_current = main_subrun_info[idx].mw
        mac_current = main_subrun_info[idx].mac
        mw_next = main_subrun_info[idx+1].mw
        mac_next = main_subrun_info[idx+1].mac
        
        mac_current_str = format_mac_latex(mac_current)
        mac_next_str = format_mac_latex(mac_next)
        
        transition_label = L"%$(method_legend_main): MW %$mw_current→%$mw_next, MAC %$mac_current_str→%$mac_next_str"
        
        vline!(p_gradient_comparison, [boundary], 
               linestyle=:dot, 
               linewidth=2,
               color=bold_hex[1],
               label=transition_label,
               alpha=0.5)
    end
end

# Add vertical lines for comparison method subrun boundaries (if looped)
if comp_num_subruns > 1
    for (idx, boundary) in enumerate(comp_iteration_boundaries[1:end-1])
        mw_current = comp_subrun_info[idx].mw
        mac_current = comp_subrun_info[idx].mac
        mw_next = comp_subrun_info[idx+1].mw
        mac_next = comp_subrun_info[idx+1].mac
        
        mac_current_str = format_mac_latex(mac_current)
        mac_next_str = format_mac_latex(mac_next)
        
        transition_label = L"%$(method_legend_compare_1): MW %$mw_current→%$mw_next, MAC %$mac_current_str→%$mac_next_str"
        
        vline!(p_gradient_comparison, [boundary], 
               linestyle=:dashdot, 
               linewidth=2,
               color=bold_hex[3],
               label=transition_label,
               alpha=0.5)
    end
end

display(p_gradient_comparison)
#save_thesis_plot(p_gradient_comparison, title_text_grad)

### Timing (Grad times, Optim times, Commu times, Total times)

In [ ]:
# Extract timing data from all methods
println("\n" * "="^90)
println("EXTRACTING TIMING DATA FROM ALL METHODS")
println("="^90)

# Dictionary structure: method_name => {nq => {timing_metric => value}}
all_methods_timing = Dict{String, Dict{Int, Dict{String, Float64}}}()

for (method_name, method_data) in all_methods_data
    println("\nMethod: $method_name")
    
    timing_by_nq = Dict{Int, Dict{String, Float64}}()
    
    # Get original data for detailed timing
    if !haskey(all_methods_original_data, method_name)
        @warn "No original data for $method_name, using loop_summary only"
        # Fall back to loop_summary data
        for nq in sort(collect(keys(method_data)))
            loop_summary = method_data[nq]
            timing_by_nq[nq] = Dict(
                "total_time" => loop_summary["total_elapsed_time_s"],
                "gradient_time" => get(loop_summary, "gradient_time_total", NaN),
                "optimization_time" => get(loop_summary, "optimization_time_total", NaN),
                "commutator_time" => get(loop_summary, "commutator_time_total", NaN)
            )
        end
    else
        # Extract from original data (sum across all subruns)
        original_data_method = all_methods_original_data[method_name]
        
        for nq in sort(collect(keys(method_data)))
            if !haskey(original_data_method, nq)
                continue
            end
            
            original_data = original_data_method[nq]
            all_results = original_data["all_results"]
            
            # all_results is an array, use length to get the last (most accurate) run
            selected_run_idx = length(all_results)
            selected_run = all_results[selected_run_idx]
            
            # Sum timing across all subruns
            total_grad_time = 0.0
            total_optim_time = 0.0
            total_comm_time = 0.0
            
            subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
            for run_idx in subrun_keys
                run_data = selected_run[run_idx]
                total_grad_time += get(run_data, "gradient_time_total", 0.0)
                total_optim_time += get(run_data, "optimization_time_total", 0.0)
                total_comm_time += get(run_data, "commutator_time_total", 0.0)
            end
            
            # Get total time from loop_summary
            loop_summary = method_data[nq]
            total_time = loop_summary["total_elapsed_time_s"]
            
            timing_by_nq[nq] = Dict(
                "total_time" => total_time,
                "gradient_time" => total_grad_time,
                "optimization_time" => total_optim_time,
                "commutator_time" => total_comm_time
            )
            
            println("  nq=$nq: Total=$(round(total_time, digits=2))s, " *
                    "Grad=$(round(total_grad_time, digits=2))s, " *
                    "Optim=$(round(total_optim_time, digits=2))s, " *
                    "Comm=$(round(total_comm_time, digits=2))s")
        end
    end
    
    all_methods_timing[method_name] = timing_by_nq
end

println("\n" * "="^90)
println("Timing data extracted for $(length(all_methods_timing)) method(s)")
println("="^90)

#### Gradient Times Comparison

In [ ]:
# Plot gradient times comparison
title_text = "Gradient Times Comparison ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_grad_times = plot(
    xlabel="Number of Qubits", 
    ylabel="Gradient Computation Time (s)",
    #yscale=:log10,
    legend=:topleft,
    size=(600, 400),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm,
    grid=true
)

for (method_idx, (method_name, timing_data)) in enumerate(all_methods_timing)
    nqs = Int[]
    grad_times = Float64[]
    
    for nq in sort(collect(keys(timing_data)))
        grad_time = timing_data[nq]["gradient_time"]
        if !isnan(grad_time) && grad_time > 0
            push!(nqs, nq)
            push!(grad_times, grad_time)
        end
    end
    
    if !isempty(nqs)
        color = bold[mod1(method_idx, length(bold))]
        scatter!(p_grad_times, nqs, grad_times,
                marker=:circle,
                color=color,
                label=method_name)
        
        plot!(p_grad_times, nqs, grad_times,
              color=color,
              label="")
    end
end

display(p_grad_times)
#save_thesis_plot(p_grad_times, title_text)

#### Optimization Times Comparison

In [ ]:
# Plot optimization times comparison
title_text = "Optimization Times Comparison ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_optim_times = plot(
    xlabel="Number of Qubits", 
    ylabel="Optimization Time (s)",
    #yscale=:log10,
    legend=:topleft,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm,
    grid=true
)

for (method_idx, (method_name, timing_data)) in enumerate(all_methods_timing)
    nqs = Int[]
    optim_times = Float64[]
    
    for nq in sort(collect(keys(timing_data)))
        optim_time = timing_data[nq]["optimization_time"]
        if !isnan(optim_time) && optim_time > 0
            push!(nqs, nq)
            push!(optim_times, optim_time)
        end
    end
    
    if !isempty(nqs)
        color = bold[mod1(method_idx, length(bold))]
        scatter!(p_optim_times, nqs, optim_times,
                marker=:square,
                color=color,
                label=method_name)
        
        plot!(p_optim_times, nqs, optim_times,
              color=color,
              label="")
    end
end

display(p_optim_times)
#save_thesis_plot(p_optim_times, title_text)

#### Commutator Times Comparison

In [ ]:
# Plot commutator times comparison
title_text = "Commutator Times Comparison ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_comm_times = plot(
    xlabel="Number of Qubits", 
    ylabel="Commutator Computation Time (s)",
    #yscale=:log10,
    legend=:topleft,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm,
    grid=true
)

for (method_idx, (method_name, timing_data)) in enumerate(all_methods_timing)
    nqs = Int[]
    comm_times = Float64[]
    
    for nq in sort(collect(keys(timing_data)))
        comm_time = timing_data[nq]["commutator_time"]
        if !isnan(comm_time) && comm_time > 0
            push!(nqs, nq)
            push!(comm_times, comm_time)
        end
    end
    
    if !isempty(nqs)
        color = bold[mod1(method_idx, length(bold))]
        scatter!(p_comm_times, nqs, comm_times,
                marker=:diamond,
                color=color,
                label=method_name)
        
        plot!(p_comm_times, nqs, comm_times,
              color=color,
              label="")
    end
end

display(p_comm_times)
#save_thesis_plot(p_comm_times, title_text)

#### Total Times Comparison

In [ ]:
# Plot total times comparison
title_text = "Total Runtime Comparison ($HAMILTONIAN_NAME)"
println("Plot: $title_text")

p_total_times = plot(
    xlabel="Number of Qubits", 
    ylabel="Total Runtime [h]",
    #yscale=:log10,
    legend=:topright,
    size=(600, 400),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm,
    grid=true
)
for (method_idx, method_name) in enumerate(method_order)
    if !haskey(all_methods_timing, method_name)
        continue
    end
    timing_data = all_methods_timing[method_name]

# for (method_idx, (method_name, timing_data)) in enumerate(all_methods_timing)
    nqs = Int[]
    total_times = Float64[]
    
    for nq in sort(collect(keys(timing_data)))

        total_time = timing_data[nq]["total_time"]
        if !isnan(total_time) && total_time > 0
            push!(nqs, nq)
            push!(total_times, total_time)
        end
    end
    
    if !isempty(nqs)
        color = bold[mod1(method_idx, length(bold))]
        scatter!(p_total_times, nqs, total_times/3600,
                marker=:circle,
                markersize=7,
                color=color,
                label=method_name)
        
        plot!(p_total_times, nqs, total_times/3600,
              color=color,
              label="")
    end
end

println("="^120)

display(p_total_times)
#save_thesis_plot(p_total_times, title_text)


### Circuit Depth vs System Size for Different ADAPT Methods

Compare how the circuit depth (number of layers/operators) required for convergence scales with system size across different ADAPT methods.

In [ ]:
# Extract circuit depth data from all methods
println("\n" * "="^90)
println("EXTRACTING CIRCUIT DEPTH DATA FROM ALL METHODS")
println("="^90)

# Dictionary structure: method_name => {nq => depth}
all_methods_depth = Dict{String, Dict{Int, Int}}()

for (method_name, method_data) in all_methods_data
    println("\nMethod: $method_name")
    
    depth_by_nq = Dict{Int, Int}()
    
    for nq in sort(collect(keys(method_data)))
        loop_summary = method_data[nq]
        
        # Get final circuit depth (total number of operators/layers)
        circuit_depth = loop_summary["final_circuit_depth"]
        depth_by_nq[nq] = circuit_depth
        
        println("  nq=$nq: depth=$circuit_depth")
    end
    
    all_methods_depth[method_name] = depth_by_nq
end

println("\n" * "="^90)
println("Circuit depth data extracted for $(length(all_methods_depth)) method(s)")
println("="^90)

# Print summary
for (method_name, depth_dict) in sort(collect(all_methods_depth), by=x->x[1])
    nqs = sort(collect(keys(depth_dict)))
    depths = [depth_dict[nq] for nq in nqs]
    println("\n$method_name:")
    println("  Qubit sizes: $nqs")
    println("  Min depth: $(minimum(depths)) (at $(nqs[argmin(depths)]) qubits)")
    println("  Max depth: $(maximum(depths)) (at $(nqs[argmax(depths)]) qubits)")
end

In [ ]:
# Plot circuit depth vs number of qubits for all ADAPT methods
if !isempty(all_methods_depth)
    title_text = "Circuit Depth vs System Size - ADAPT Method Comparison"
    println("Plot: $title_text")
    
    # Create main plot
    p_depth = plot(
        xlabel="Number of Qubits",
        ylabel="Circuit Depth",
        legend=:topleft,
        size=(600, 400),
        left_margin=5Plots.mm,
        bottom_margin=5Plots.mm,
        grid=true
    )
    
    # Plot each ADAPT method in the same order as the total runtime plot
    for (idx, method_name) in enumerate(method_order)
        if !haskey(all_methods_depth, method_name)
            continue
        end
        depth_dict = all_methods_depth[method_name]
        color = bold[mod1(idx, length(bold))]
        
        # Get sorted qubit sizes and corresponding depths
        nqs = sort(collect(keys(depth_dict)))
        depths = [depth_dict[nq] for nq in nqs]
        
        # Plot with markers and lines
        plot!(p_depth, nqs, depths,
              label=method_name,
              marker=:circle,
              markersize=5,
              linewidth=2,
              color=color)
    end
    
    display(p_depth)
    #save_thesis_plot(p_depth, title_text)
    
else
    @warn "No depth data available for plotting"
end


## Memory Analysis

In [ ]:
# Memory Analysis: Extract memory data from loaded runs
println("\n" * "="^80)
println("MEMORY DATA AVAILABLE IN LOADED RUNS")
println("="^80)

# Check what memory fields are available by looking at one example
example_nq = sort(collect(keys(hamiltonian_loop_runs)))[1] # TODO hardcoded run count!
loop_data = hamiltonian_loop_runs[example_nq]

println("Checking memory fields in loop_summary for nq=$example_nq:")
memory_fields = filter(k -> contains(lowercase(string(k)), "memory") || contains(lowercase(string(k)), "rss"), keys(loop_data))
for field in sort(collect(memory_fields))
    println("  - $field: $(loop_data[field])")
end
println("="^80)

### Total Memory vs nq Count

Plot the total memory usage (memory_gb) for each system size.

In [ ]:
# Extract total memory data for all nq
all_nq_values = sort(collect(keys(hamiltonian_loop_runs)))
total_memory_gb = Float64[]
total_memory_mb = Float64[]
peak_rss_mb = Float64[]

println("\n" * "="^80)
println("TOTAL MEMORY USAGE BY SYSTEM SIZE")
println("="^80)
@printf "%-10s %-20s %-20s %-20s\n" "nq" "Total Mem (GB)" "Total Mem (MB)" "Peak RSS (MB)"
println("-"^80)

for nq in all_nq_values
    loop_data = hamiltonian_loop_runs[nq]
    
    mem_gb = haskey(loop_data, "total_memory_gb") ? loop_data["total_memory_gb"] : NaN
    mem_mb = haskey(loop_data, "total_memory_bytes") ? loop_data["total_memory_bytes"] / 1e6 : NaN
    peak_mb = haskey(loop_data, "peak_rss_mb") ? loop_data["peak_rss_mb"] : NaN
    
    push!(total_memory_gb, mem_gb)
    push!(total_memory_mb, mem_mb)
    push!(peak_rss_mb, peak_mb)
    
    @printf "%-10d %-20.6f %-20.2f %-20.2f\n" nq mem_gb mem_mb peak_mb
end
println("="^80)

In [ ]:
# Plot Total Memory vs nq
p_memory_total = plot(
    xlabel="Number of Qubits (nq)",
    ylabel="Total Memory (GB)",
    title="Total Memory Allocated vs System Size - $HAMILTONIAN_NAME",
    legend=:topleft,
    grid=true,
    minorticks=true,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Filter out NaN values for plotting
valid_indices = .!isnan.(total_memory_gb)
valid_nq = all_nq_values[valid_indices]
valid_memory = total_memory_gb[valid_indices]

if !isempty(valid_nq)
    plot!(p_memory_total, valid_nq, valid_memory,
          marker=:circle,
          markersize=8,
          linewidth=2,
          color=:blue,
          label="Total Memory Allocated")
    
    display(p_memory_total)
else
    println("No valid memory data to plot")
end

In [ ]:
# Plot Peak RSS vs nq (separate plot)
p_peak_rss = plot(
    xlabel="Number of Qubits (nq)",
    ylabel="Peak RSS (GB)",
    title="Peak Resident Set Size vs System Size - $HAMILTONIAN_NAME",
    legend=:topleft,
    grid=true,
    minorticks=true,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Filter out NaN values for plotting
valid_indices_rss = .!isnan.(peak_rss_mb)
valid_nq_rss = all_nq_values[valid_indices_rss]
valid_rss_gb = peak_rss_mb[valid_indices_rss] ./ 1024.0  # Convert to GB

if !isempty(valid_nq_rss)
    plot!(p_peak_rss, valid_nq_rss, valid_rss_gb,
          marker=:diamond,
          markersize=8,
          linewidth=2,
          color=:red,
          label="Peak RSS")
    
    display(p_peak_rss)
else
    println("No valid Peak RSS data to plot")
end

### Memory Per Iteration Analysis (from Subruns)

Extract memory per iteration from the subruns in `original_data_by_nq` to get the actual iteration-by-iteration memory usage.

In [ ]:
"""
Extract per-iteration memory data from subruns for a specific nq.

Returns:
- all_iterations: Iteration numbers
- gradient_memory_gb: Gradient calculation memory per iteration (GB)
- commutator_memory_gb: Commutator calculation memory per iteration (GB)  
- optimization_memory_gb: Optimization memory per iteration (GB)
- total_memory_gb: Total memory per iteration (GB)
"""
function extract_memory_per_iteration(nq::Int, original_data_by_nq::Dict)
    if !haskey(original_data_by_nq, nq)
        @warn "No original data found for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    original_data = original_data_by_nq[nq]
    all_results = original_data["all_results"]
    
    # NEW: all_results is now a Dict, not an Array
    # Get the last/highest run number
    run_keys = sort(collect(keys(all_results)))
    if isempty(run_keys)
        @warn "No runs found in all_results for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    selected_run_idx = run_keys[end]  # Use the last (most accurate) run
    selected_run = all_results[selected_run_idx]
    
    # Get all subrun keys (should be integers)
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    if isempty(subrun_keys)
        @warn "No subruns found for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    # Collect data from each subrun
    all_iterations = Int[]
    all_gradient_memory_gb = Float64[]
    all_commutator_memory_gb = Float64[]
    all_optimization_memory_gb = Float64[]
    all_total_memory_gb = Float64[]
    iteration_offset = 0
    
    for subrun_idx in subrun_keys
        subrun = selected_run[subrun_idx]
        
        # NEW: Check for num_iterations or energy_curve
        n_iters = 0
        if haskey(subrun, "num_iterations")
            n_iters = subrun["num_iterations"]
        elseif haskey(subrun, "energy_curve")
            n_iters = length(subrun["energy_curve"])
        else
            # Skip this subrun if we can't determine iterations
            continue
        end
        
        # Get memory arrays for this subrun (in bytes, convert to GB)
        gradient_mem = haskey(subrun, "gradient_memory") ? subrun["gradient_memory"] : Float64[]
        commutator_mem = haskey(subrun, "commutator_memory") ? subrun["commutator_memory"] : Float64[]
        optimization_mem = haskey(subrun, "optimization_memory") ? subrun["optimization_memory"] : Float64[]
        
        # Ensure we have data for all iterations
        if isempty(gradient_mem)
            # If memory tracking wasn't available, fill with NaN
            gradient_mem = fill(NaN, n_iters)
            commutator_mem = fill(NaN, n_iters)
            optimization_mem = fill(NaN, n_iters)
        end
        
        # Use the minimum of n_iters and actual array lengths to avoid bounds errors
        actual_iters = min(n_iters, length(gradient_mem), length(commutator_mem), length(optimization_mem))
        
        # Store for each iteration in this subrun
        for i in 1:actual_iters
            push!(all_iterations, iteration_offset + i)
            # Convert from bytes to GB
            grad_gb = gradient_mem[i] / 1e9
            comm_gb = commutator_mem[i] / 1e9
            opt_gb = optimization_mem[i] / 1e9
            push!(all_gradient_memory_gb, grad_gb)
            push!(all_commutator_memory_gb, comm_gb)
            push!(all_optimization_memory_gb, opt_gb)
            push!(all_total_memory_gb, grad_gb + comm_gb + opt_gb)
        end
        
        # Use actual_iters for offset instead of n_iters
        iteration_offset += actual_iters
    end
    
    return all_iterations, all_gradient_memory_gb, all_commutator_memory_gb, all_optimization_memory_gb, all_total_memory_gb
end

### Plot Memory Per Iteration for Specific System Size

Extract and plot per-iteration memory usage (gradient, commutator, optimization) for a specific nq value.

In [ ]:
# Choose an nq to analyze (e.g., 10 qubits)
target_nq = 80

println("\n" * "="^80)
println("EXTRACTING MEMORY PER ITERATION FOR nq=$target_nq")
println("="^80)

# Use the new data structure: all_methods_original_data[method_legend_main]
iterations, gradient_mem_gb, commutator_mem_gb, optimization_mem_gb, total_mem_gb = extract_memory_per_iteration(target_nq, all_methods_original_data[method_legend_main])

if !isempty(iterations)
    println("Found $(length(iterations)) iterations")
    
    # Filter out NaN values for statistics
    valid_total = filter(!isnan, total_mem_gb)
    if !isempty(valid_total)
        println("Total memory range: $(minimum(valid_total)) - $(maximum(valid_total)) GB")
    else
        println("⚠ All memory values are NaN - data may not have been tracked in your runs")
    end
    
    # Show first few and last few values
    n_show = min(5, length(iterations))
    println("\nFirst $n_show iterations:")
    for i in 1:n_show
        println("  Iter $(iterations[i]): Grad=$(round(gradient_mem_gb[i], digits=3)) GB, " *
                "Comm=$(round(commutator_mem_gb[i], digits=3)) GB, " *
                "Opt=$(round(optimization_mem_gb[i], digits=3)) GB, " *
                "Total=$(round(total_mem_gb[i], digits=3)) GB")
    end
    
    if length(iterations) > 2*n_show
        println("  ...")
        println("Last $n_show iterations:")
        for i in (length(iterations)-n_show+1):length(iterations)
            println("  Iter $(iterations[i]): Grad=$(round(gradient_mem_gb[i], digits=3)) GB, " *
                    "Comm=$(round(commutator_mem_gb[i], digits=3)) GB, " *
                    "Opt=$(round(optimization_mem_gb[i], digits=3)) GB, " *
                    "Total=$(round(total_mem_gb[i], digits=3)) GB")
        end
    end
else
    println("No iteration data found!")
end
println("="^80)

In [ ]:
# Debug: Check what's in the subrun
if haskey(all_methods_original_data, method_legend_main) && haskey(all_methods_original_data[method_legend_main], target_nq)
    data = all_methods_original_data[method_legend_main][target_nq]
    all_results = data["all_results"]
    
    run_keys = sort(collect(keys(all_results)))
    selected_run = all_results[run_keys[end]]
    
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    if !isempty(subrun_keys)
        first_subrun = selected_run[subrun_keys[1]]
        println("First subrun (key=$(subrun_keys[1])):")
        println("  Keys: ", collect(keys(first_subrun)))
        println("  Has 'energy_curve'? ", haskey(first_subrun, "energy_curve"))
        println("  Has 'gradient_memory'? ", haskey(first_subrun, "gradient_memory"))
        println("  Has 'commutator_memory'? ", haskey(first_subrun, "commutator_memory"))
        println("  Has 'optimization_memory'? ", haskey(first_subrun, "optimization_memory"))
        println("  Has 'rss_per_iteration'? ", haskey(first_subrun, "rss_per_iteration"))
        
        if haskey(first_subrun, "energy_curve")
            println("  Length of energy_curve: ", length(first_subrun["energy_curve"]))
        end
        
        # Check rss_per_iteration in detail
        if haskey(first_subrun, "rss_per_iteration")
            rss_data = first_subrun["rss_per_iteration"]
            println("  rss_per_iteration type: ", typeof(rss_data))
            println("  rss_per_iteration length: ", length(rss_data))
            if !isempty(rss_data)
                println("  First few values: ", rss_data[1:min(5, length(rss_data))])
            else
                println("  ⚠ rss_per_iteration is EMPTY!")
            end
        else
            println("  ⚠ rss_per_iteration not found in subrun!")
        end
        
        # Check for other RSS-related fields
        println("\n  Checking for other RSS fields:")
        for key in keys(first_subrun)
            if occursin("rss", lowercase(string(key))) || occursin("memory", lowercase(string(key)))
                val = first_subrun[key]
                println("    $key: type=$(typeof(val)), length=$(length(val))")
                if !isempty(val) && length(val) <= 5
                    println("      Values: $val")
                elseif !isempty(val)
                    println("      First 3: $(val[1:min(3,length(val))])")
                end
            end
        end
        
        # Check if there's RSS data at the run level (not subrun level)
        println("\n  Checking run-level keys:")
        run_level_keys = filter(k -> !isa(k, Integer), keys(selected_run))
        for key in run_level_keys
            if occursin("rss", lowercase(string(key))) || occursin("peak", lowercase(string(key)))
                println("    Found at run level: $key")
                val = selected_run[key]
                println("      Type: $(typeof(val)), Value: $val")
            end
        end
    end
end

In [ ]:
# Plot memory per iteration breakdown for the selected nq
title_text = "Memory per Iteration Breakdown for nq=$target_nq - $HAMILTONIAN_NAME"
println("Plot: $title_text")
if !isempty(iterations) && !all(isnan, total_mem_gb)
    p_memory_breakdown = plot(
        xlabel="ADAPT Iteration",
        ylabel="Memory [GB]",

        legend=:topleft,
        grid=true,
        minorticks=true,
        size=(600, 400),
        left_margin=10Plots.mm,
        bottom_margin=6Plots.mm
    )
    
    # Plot each component
    plot!(p_memory_breakdown, iterations, gradient_mem_gb,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[1],
          label="Gradient Memory")
    
    plot!(p_memory_breakdown, iterations, commutator_mem_gb,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[2],
          label="Commutator Memory")
    
    plot!(p_memory_breakdown, iterations, optimization_mem_gb,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[3],
          label="Optimization Memory")
    
    plot!(p_memory_breakdown, iterations, total_mem_gb,
          marker=:circle,
          markersize=4,
          linewidth=3,
          color=bold_hex[4],
          linestyle=:dash,
          label="Total Memory")
elseif !isempty(iterations)
    println("⚠ Memory data contains only NaN values - tracking may not have been enabled during your runs")
else
    println("No iteration data found for nq=$target_nq")
end
display(p_memory_breakdown)
#save_thesis_plot(p_memory_breakdown, title_text)


In [ ]:
# Extract and plot peak RSS memory per iteration for the selected nq
title_text = "Peak RSS Memory per Iteration for nq=$target_nq - $HAMILTONIAN_NAME"
println("Plot: $title_text")

# Extract rss_per_iteration data
iterations_rss = Int[]
rss_per_iter_mb = Float64[]

if haskey(all_methods_original_data, method_legend_main) && haskey(all_methods_original_data[method_legend_main], target_nq)
    data = all_methods_original_data[method_legend_main][target_nq]
    all_results = data["all_results"]
    
    # Get the most recent run
    run_keys = sort(collect(keys(all_results)))
    selected_run = all_results[run_keys[end]]
    
    # Get all subrun indices (iterations)
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    for subrun_idx in subrun_keys
        subrun = selected_run[subrun_idx]
        
        # Extract rss_per_iteration array if available
        if haskey(subrun, "rss_per_iteration")
            rss_iter = get(subrun, "rss_per_iteration", Float64[])
            if !isempty(rss_iter)
                # Each subrun has its own rss_per_iteration array
                for (idx, rss_val) in enumerate(rss_iter)
                    push!(iterations_rss, length(iterations_rss) + 1)
                    push!(rss_per_iter_mb, rss_val)
                end
            end
        end
    end
end

# Convert to GB
rss_per_iter_gb = rss_per_iter_mb ./ 1024.0

if !isempty(iterations_rss) && !all(isnan, rss_per_iter_gb)
    p_rss_memory = plot(
        xlabel="ADAPT Iteration",
        ylabel="Peak RSS Memory [GB]",
        legend=:topleft,
        grid=true,
        minorticks=true,
        size=(600, 400),
        left_margin=10Plots.mm,
        bottom_margin=6Plots.mm
    )
    
    # Plot peak RSS memory over iterations
    plot!(p_rss_memory, iterations_rss, rss_per_iter_gb,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[1],
          label="Peak RSS Memory")
    
    display(p_rss_memory)
    #save_thesis_plot(p_rss_memory, title_text)
    
    # Show statistics
    println("\nPeak RSS Memory Statistics:")
    println("  Min: $(round(minimum(rss_per_iter_gb), digits=3)) GB")
    println("  Max: $(round(maximum(rss_per_iter_gb), digits=3)) GB")
    println("  Mean: $(round(mean(rss_per_iter_gb), digits=3)) GB")
    println("  Total iterations: $(length(iterations_rss))")
elseif !isempty(iterations_rss)
    println("⚠ Peak RSS memory data contains only NaN values - tracking may not have been enabled during your runs")
else
    println("No rss_per_iteration data found for nq=$target_nq")
end

### Compare Memory Per Iteration for Multiple System Sizes

Plot total memory evolution for several nq values on the same plot.

In [ ]:
# Compare memory per iteration for multiple nq values
nq_values_to_compare = [40,50]  # Adjust as needed

p_memory_compare = plot(
    xlabel="ADAPT Iteration",
    ylabel="Total Memory per Iteration (GB)",
    title="Total Memory per Iteration - Multiple System Sizes - $HAMILTONIAN_NAME",
    legend=:topright,
    grid=true,
    minorticks=true,
    size=(1200, 700),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

colors = [:blue, :red, :green, :orange, :purple, :brown, :pink, :gray]

# Use the new data structure for memory analysis
original_data = all_methods_original_data[method_legend_main]

for (idx, nq) in enumerate(nq_values_to_compare)
    if haskey(original_data, nq)
        iterations, _, _, _, total_mem_gb = extract_memory_per_iteration(nq, original_data)
        
        if !isempty(iterations) && !all(isnan, total_mem_gb)
            color_idx = ((idx - 1) % length(colors)) + 1
            
            plot!(p_memory_compare, iterations, total_mem_gb,
                  marker=:circle,
                  markersize=3,
                  linewidth=2,
                  color=colors[color_idx],
                  label="nq=$nq")
        end
    end
end

display(p_memory_compare)

In [ ]:
# Compute average memory per iteration for each system size
original_data = all_methods_original_data[method_legend_main]
all_nq_values = sort(collect(keys(original_data)))

memory_per_iter_data = Float64[]

println("\nComputing average memory per iteration for each system size:")
for nq in all_nq_values
    iterations, _, _, _, total_mem_gb = extract_memory_per_iteration(nq, original_data)
    
    if !isempty(iterations) && !all(isnan, total_mem_gb)
        valid_mem = filter(!isnan, total_mem_gb)
        if !isempty(valid_mem)
            avg_mem_gb = mean(valid_mem)
            avg_mem_mb = avg_mem_gb * 1000  # Convert to MB
            push!(memory_per_iter_data, avg_mem_mb)
            println("  nq=$nq: $(round(avg_mem_mb, digits=2)) MB/iter ($(length(valid_mem)) iterations)")
        else
            push!(memory_per_iter_data, NaN)
            println("  nq=$nq: No valid memory data")
        end
    else
        push!(memory_per_iter_data, NaN)
        println("  nq=$nq: No iteration data")
    end
end

# Plot Memory Per Iteration vs nq
p_memory_per_iter = plot(
    xlabel="Number of Qubits (nq)",
    ylabel="Average Memory per Iteration (MB)",
    title="Memory per ADAPT Iteration vs System Size - $HAMILTONIAN_NAME",
    legend=:topleft,
    grid=true,
    minorticks=true,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Filter out NaN values for plotting
valid_indices_iter = .!isnan.(memory_per_iter_data)
valid_nq_iter = all_nq_values[valid_indices_iter]
valid_memory_iter = memory_per_iter_data[valid_indices_iter]

if !isempty(valid_nq_iter)
    plot!(p_memory_per_iter, valid_nq_iter, valid_memory_iter,
          marker=:circle,
          markersize=8,
          linewidth=2,
          color=:purple,
          label="Avg Memory/Iteration")
    
    display(p_memory_per_iter)
else
    println("\nNo valid memory per iteration data to plot")
end

## Timing Analysis

Analyze the timing behavior of the ADAPT-VQE algorithm across different system sizes and iterations.

## Total Time vs System Size

Plot the total elapsed time for each system size.

In [ ]:
# Extract total time for each system size from Main Method
original_data = all_methods_original_data[method_legend_main]
all_nq_values = sort(collect(keys(original_data)))

total_time_data = Float64[]

println("\n" * "="^80)
println("EXTRACTING TOTAL TIME FOR EACH SYSTEM SIZE")
println("="^80)

for nq in all_nq_values
    if haskey(hamiltonian_loop_runs, nq)
        loop_summary = hamiltonian_loop_runs[nq]
        
        if haskey(loop_summary, "total_elapsed_time_s")
            total_time_s = loop_summary["total_elapsed_time_s"]
            push!(total_time_data, total_time_s)
            println("nq=$nq: $(round(total_time_s, digits=2)) seconds")
        else
            push!(total_time_data, NaN)
            println("nq=$nq: No timing data available")
        end
    else
        push!(total_time_data, NaN)
        println("nq=$nq: No loop summary found")
    end
end

println("="^80)

# Plot Total Time vs nq
p_total_time = plot(
    xlabel="Number of Qubits (nq)",
    ylabel="Total Time (seconds)",
    title="Total ADAPT-VQE Runtime vs System Size - $HAMILTONIAN_NAME",
    legend=:topleft,
    grid=true,
    minorticks=true,
    size=(1000, 600),
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Filter out NaN values for plotting
valid_indices = .!isnan.(total_time_data)
valid_nq = all_nq_values[valid_indices]
valid_time = total_time_data[valid_indices]

if !isempty(valid_nq)
    plot!(p_total_time, valid_nq, valid_time,
          marker=:circle,
          markersize=8,
          linewidth=2,
          color=:blue,
          label="Total Runtime")
    
    display(p_total_time)
else
    println("No valid timing data to plot")
end

## Time Per Iteration Analysis (from Subruns)

Extract time per iteration from the subruns to get the actual iteration-by-iteration timing breakdown.

In [ ]:
"""
Extract per-iteration timing data from subruns for a specific nq.

Returns:
- all_iterations: Iteration numbers
- gradient_time_s: Gradient calculation time per iteration (seconds)
- commutator_time_s: Commutator calculation time per iteration (seconds)  
- optimization_time_s: Optimization time per iteration (seconds)
- total_time_s: Total time per iteration (seconds)
"""
function extract_time_per_iteration(nq::Int, original_data_by_nq::Dict)
    if !haskey(original_data_by_nq, nq)
        @warn "No original data found for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    original_data = original_data_by_nq[nq]
    all_results = original_data["all_results"]
    
    # Get the last/highest run number
    run_keys = sort(collect(keys(all_results)))
    if isempty(run_keys)
        @warn "No runs found in all_results for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    selected_run_idx = run_keys[end]  # Use the last (most accurate) run
    selected_run = all_results[selected_run_idx]
    
    # Get all subrun keys (should be integers)
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    if isempty(subrun_keys)
        @warn "No subruns found for nq=$nq"
        return Int[], Float64[], Float64[], Float64[], Float64[]
    end
    
    # Collect data from each subrun
    all_iterations = Int[]
    all_gradient_time_s = Float64[]
    all_commutator_time_s = Float64[]
    all_optimization_time_s = Float64[]
    all_total_time_s = Float64[]
    iteration_offset = 0
    
    for subrun_idx in subrun_keys
        subrun = selected_run[subrun_idx]
        
        # Check for num_iterations or energy_curve
        n_iters = 0
        if haskey(subrun, "num_iterations")
            n_iters = subrun["num_iterations"]
        elseif haskey(subrun, "energy_curve")
            n_iters = length(subrun["energy_curve"])
        else
            # Skip this subrun if we can't determine iterations
            continue
        end
        
        # Get timing arrays for this subrun (in seconds)
        gradient_time = haskey(subrun, "gradient_times") ? subrun["gradient_times"] : Float64[]
        commutator_time = haskey(subrun, "commutator_times") ? subrun["commutator_times"] : Float64[]
        optimization_time = haskey(subrun, "optimization_times") ? subrun["optimization_times"] : Float64[]
        
        # Ensure we have data for all iterations
        if isempty(gradient_time)
            # If timing tracking wasn't available, fill with NaN
            gradient_time = fill(NaN, n_iters)
            commutator_time = fill(NaN, n_iters)
            optimization_time = fill(NaN, n_iters)
        end
        
        # Use the minimum of n_iters and actual array lengths to avoid bounds errors
        actual_iters = min(n_iters, length(gradient_time), length(commutator_time), length(optimization_time))
        
        # Store for each iteration in this subrun
        for i in 1:actual_iters
            push!(all_iterations, iteration_offset + i)
            grad_s = gradient_time[i]
            comm_s = commutator_time[i]
            opt_s = optimization_time[i]
            push!(all_gradient_time_s, grad_s)
            push!(all_commutator_time_s, comm_s)
            push!(all_optimization_time_s, opt_s)
            push!(all_total_time_s, grad_s + comm_s + opt_s)
        end
        
        # Use actual_iters for offset instead of n_iters
        iteration_offset += actual_iters
    end
    
    return all_iterations, all_gradient_time_s, all_commutator_time_s, all_optimization_time_s, all_total_time_s
end

println("✓ Timing extraction function defined")

### Plot Time Per Iteration for Specific System Size

Extract and plot per-iteration timing (gradient, commutator, optimization) for a specific nq value.

In [ ]:
# Choose an nq to analyze (e.g., 50 qubits)
target_nq_time = 80

println("\n" * "="^80)
println("EXTRACTING TIME PER ITERATION FOR nq=$target_nq_time")
println("="^80)

# Use the new data structure: all_methods_original_data[method_legend_main]
iterations, gradient_time_s, commutator_time_s, optimization_time_s, total_time_s = extract_time_per_iteration(target_nq_time, all_methods_original_data[method_legend_main])

if !isempty(iterations)
    println("Found $(length(iterations)) iterations")
    
    # Filter out NaN values for statistics
    valid_total = filter(!isnan, total_time_s)
    if !isempty(valid_total)
        println("Total time range: $(minimum(valid_total)) - $(maximum(valid_total)) seconds")
        println("Total accumulated time: $(round(sum(valid_total), digits=2)) seconds")
    else
        println("⚠ All time values are NaN - data may not have been tracked in your runs")
    end
    
    # Show first few and last few values
    n_show = min(5, length(iterations))
    println("\nFirst $n_show iterations:")
    for i in 1:n_show
        println("  Iter $(iterations[i]): Grad=$(round(gradient_time_s[i], digits=3)) s, " *
                "Comm=$(round(commutator_time_s[i], digits=3)) s, " *
                "Opt=$(round(optimization_time_s[i], digits=3)) s, " *
                "Total=$(round(total_time_s[i], digits=3)) s")
    end
    
    if length(iterations) > 2*n_show
        println("  ...")
        println("Last $n_show iterations:")
        for i in (length(iterations)-n_show+1):length(iterations)
            println("  Iter $(iterations[i]): Grad=$(round(gradient_time_s[i], digits=3)) s, " *
                    "Comm=$(round(commutator_time_s[i], digits=3)) s, " *
                    "Opt=$(round(optimization_time_s[i], digits=3)) s, " *
                    "Total=$(round(total_time_s[i], digits=3)) s")
        end
    end
else
    println("No iteration data found!")
end
println("="^80)

In [ ]:
# Debug: Check what timing data is in the subrun
if haskey(all_methods_original_data, method_legend_main) && haskey(all_methods_original_data[method_legend_main], target_nq_time)
    data = all_methods_original_data[method_legend_main][target_nq_time]
    all_results = data["all_results"]
    
    run_keys = sort(collect(keys(all_results)))
    selected_run = all_results[run_keys[end]]
    
    subrun_keys = sort([k for k in keys(selected_run) if k isa Integer])
    
    if !isempty(subrun_keys)
        first_subrun = selected_run[subrun_keys[1]]
        println("First subrun (key=$(subrun_keys[1])):")
        println("  Keys: ", collect(keys(first_subrun)))
        println("  Has 'gradient_times'? ", haskey(first_subrun, "gradient_times"))
        println("  Has 'commutator_times'? ", haskey(first_subrun, "commutator_times"))
        println("  Has 'optimization_times'? ", haskey(first_subrun, "optimization_times"))
        
        if haskey(first_subrun, "gradient_times")
            println("  Length of gradient_times: ", length(first_subrun["gradient_times"]))
        end
    end
end

In [ ]:
title_text = "Time per Iteration Breakdown for nq=$target_nq_time - $HAMILTONIAN_NAME"
# Plot time per iteration breakdown for the selected nq
if !isempty(iterations) && !all(isnan, total_time_s)
    p_time_breakdown = plot(
        xlabel="ADAPT Iteration",
        ylabel="Time [s]",
        legend=:topleft,
        grid=true,
        minorticks=true,
        size=(600, 400),
        left_margin=10Plots.mm,
        bottom_margin=6Plots.mm
    )
    
    # Plot each component
    plot!(p_time_breakdown, iterations, gradient_time_s,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[1],
          label="Gradient Time")
    
    plot!(p_time_breakdown, iterations, commutator_time_s,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[2],
          label="Commutator Time")
    
    plot!(p_time_breakdown, iterations, optimization_time_s,
          marker=:circle,
          markersize=4,
          linewidth=2,
          color=bold_hex[3],
          label="Optimization Time")
    
    plot!(p_time_breakdown, iterations, total_time_s,
          marker=:circle,
          markersize=4,
          linewidth=3,
          color=bold_hex[4],
          linestyle=:dash,
          label="Total Time")

elseif !isempty(iterations)
    println("⚠ Time data contains only NaN values - tracking may not have been enabled during your runs")
else
    println("No iteration data found for nq=$target_nq_time")
end
display(p_time_breakdown)
save_thesis_plot(p_time_breakdown, title_text)

## Observable Evaluations

### Magnetization Check: All Directions (Even N)

For even N, the ground state should be a global singlet, which means the magnetization in all directions should be equal: $\langle X_i \rangle = \langle Y_i \rangle = \langle Z_i \rangle$.

We'll calculate the total magnetization in each direction by propagating the observables through the circuit and plot the results vs. nq.

In [ ]:
# Set truncation parameters for re-evaluation
symmetry_check_mw = 20
symmetry_check_mac = 1e-7

println("="^80)
println("MAGNETIZATION SYMMETRY CHECK CONFIGURATION")
println("="^80)
println("Truncation for observable propagation:")
println("  max_weight: $symmetry_check_mw")
println("  min_abs_coeff: $symmetry_check_mac")
println("="^80)

In [ ]:
"""
    calculate_magnetization_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)

Calculate magnetization in all three directions (X, Y, Z) by propagating observables.

Returns:
- M_x: Total magnetization in X direction
- M_y: Total magnetization in Y direction  
- M_z: Total magnetization in Z direction
"""
function calculate_magnetization_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)
    # Build total magnetization operators for all three directions
    M_x = PauliSum(nq)
    M_y = PauliSum(nq)
    M_z = PauliSum(nq)
    
    for i in 1:nq
        add!(M_x, :X, i)
        add!(M_y, :Y, i)
        add!(M_z, :Z, i)
    end
    
    # Propagate observables through circuit (backpropagation)
    M_x_prop = propagate!(circuit, M_x, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    M_y_prop = propagate!(circuit, M_y, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    M_z_prop = propagate!(circuit, M_z, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    
    # Calculate expectation values using the correct overlap function
    # For Heisenberg/J1J2, reference state is |Neel⟩
    # For TFIM, reference state is |+⟩
    magn_x = overlap_function(M_x_prop, nq)
    magn_y = overlap_function(M_y_prop, nq)
    magn_z = overlap_function(M_z_prop, nq)
    
    return magn_x, magn_y, magn_z
end

println("✓ Magnetization calculation function defined");

In [ ]:
# Calculate magnetization for all even N systems
magnetization_results_xyz = Dict{Int, Tuple{Float64, Float64, Float64}}()

# Filter for even nq values
even_nq_values = sort([nq for nq in keys(hamiltonian_loop_runs) if nq % 2 == 0])

println("\n" * "="^80)
println("CALCULATING MAGNETIZATION IN ALL DIRECTIONS (EVEN N ONLY)")
println("="^80)
println("Even qubit counts: $even_nq_values")
println("Truncation: max_weight=$symmetry_check_mw, min_abs_coeff=$symmetry_check_mac")
println("="^80)

for nq in even_nq_values
    println("\n--- Processing nq = $nq ---")
    
    loop_summary = hamiltonian_loop_runs[nq]
    
    # Get circuit and optimized parameters
    circuit = loop_summary["circuit"]
    thetas = loop_summary["optimized_thetas"]
    
    println("Circuit depth: $(length(circuit)), Parameters: $(length(thetas))")
    
    try
        # Calculate magnetization in all three directions
        Mx, My, Mz = calculate_magnetization_xyz(circuit, thetas, nq; 
                                                   max_weight=symmetry_check_mw, 
                                                   min_abs_coeff=symmetry_check_mac)
        
        # Store results
        magnetization_results_xyz[nq] = (Mx, My, Mz)
        
        @printf "  M_x = %+.8f\n" Mx
        @printf "  M_y = %+.8f\n" My
        @printf "  M_z = %+.8f\n" Mz
        @printf "  Max difference: %.8e\n" maximum([abs(Mx-My), abs(Mx-Mz), abs(My-Mz)])
        
    catch e
        @warn "Failed to calculate magnetization for nq=$nq: $e"
    end
end

println("\n" * "="^80)
println("Magnetization calculation complete for $(length(magnetization_results_xyz)) even-N systems")
println("="^80)

In [ ]:
# Plot magnetization violations (deviations from 0) for all three directions
if !isempty(magnetization_results_xyz)
    sorted_nqs = sort(collect(keys(magnetization_results_xyz)))
    
    Mx_vals = [magnetization_results_xyz[nq][1] for nq in sorted_nqs]
    My_vals = [magnetization_results_xyz[nq][2] for nq in sorted_nqs]
    Mz_vals = [magnetization_results_xyz[nq][3] for nq in sorted_nqs]
    
    # Replace zeros with machine precision (10^-16) for plotting
    Mx_abs = [abs(val) < 1e-16 ? 1e-16 : abs(val) for val in Mx_vals]
    My_abs = [abs(val) < 1e-16 ? 1e-16 : abs(val) for val in My_vals]
    Mz_abs = abs.(Mz_vals)
    
    title_text = "Magnetization Symmetry Violations (Even N) - $HAMILTONIAN_NAME"
    p_mag = plot(
        xlabel="Number of Qubits",
        ylabel="Absolute Error",
        legend=(0.05,0.5),
        grid=true,
        #xlim = [4,70],
        yticks=([1e-12, 1e-8, 1e-4, 1e-2, 1e-1], 
        [L"10^{-12}", L"10^{-8}",L"10^{-4}", L"10^{-2}", L"10^{-1}"]),
        yaxis=:log,
        size=(600, 400),
        framestyle=:box,
        left_margin=8Plots.mm,
        bottom_margin=6Plots.mm
    )
    
    # Plot absolute deviations for all three magnetization components
    plot!(p_mag, sorted_nqs, Mx_abs,
          label=L"|M_x|",
          marker=:circle,
          markersize=5,
          linewidth=2,
          color=bold_hex[1])
    
    plot!(p_mag, sorted_nqs, My_abs,
          label=L"|M_y|",
          marker=:circle,
          markersize=5,
          linewidth=2,
          linestyle=:dash,
          color=bold_hex[2])
    
    plot!(p_mag, sorted_nqs, Mz_abs,
          label=L"|M_z|",
          marker=:circle,
          markersize=5,
          linewidth=2,
          color=bold_hex[3])
    
    # Add horizontal line at machine precision for reference
    hline!(p_mag, [1e-16], 
           linestyle=:dash, 
           linewidth=1, 
           color=:gray, 
           label=L"\textrm{Machine precision}\ (10^{-16})",
           alpha=0.5)

    
    # Print summary statistics
    println("\n" * "="^80)
    println("MAGNETIZATION SYMMETRY STATISTICS")
    println("="^80)
    @printf "%-8s %-15s %-15s %-15s %-15s\n" "nq" "⟨M_x⟩" "⟨M_y⟩" "⟨M_z⟩" "Max Diff"
    println("-"^80)
    
    for nq in sorted_nqs
        Mx, My, Mz = magnetization_results_xyz[nq]
        max_diff = maximum([abs(Mx-My), abs(Mx-Mz), abs(My-Mz)])
        @printf "%-8d %-15.8f %-15.8f %-15.8f %-15.8e\n" nq Mx My Mz max_diff
    end
    
    println("="^80)
    
    # Overall statistics
    all_max_diffs = [maximum([abs(magnetization_results_xyz[nq][1]-magnetization_results_xyz[nq][2]), 
                               abs(magnetization_results_xyz[nq][1]-magnetization_results_xyz[nq][3]), 
                               abs(magnetization_results_xyz[nq][2]-magnetization_results_xyz[nq][3])]) 
                     for nq in sorted_nqs]
    
    println("\nSymmetry violation statistics:")
    println("  Mean max difference: $(mean(all_max_diffs))")
    println("  Median max difference: $(median(all_max_diffs))")
    println("  Largest max difference: $(maximum(all_max_diffs)) (at nq=$(sorted_nqs[argmax(all_max_diffs)]))")
    println("  Smallest max difference: $(minimum(all_max_diffs)) (at nq=$(sorted_nqs[argmin(all_max_diffs)]))")
    println("="^80)
display(p_mag)
#save_thesis_plot(p_mag, title_text)
else
    @warn "No magnetization data available for plotting"
end


### Isotropy Check: Spin-Spin Correlations at Chain Center

For systems with SU(2) symmetry, spin-spin correlations should be isotropic: $\langle S_i^x S_j^x \rangle = \langle S_i^y S_j^y \rangle = \langle S_i^z S_j^z \rangle$.

We calculate:
- **Nearest Neighbor (NN)**: Correlations between adjacent sites at the middle of the chain
- **Next-Nearest Neighbor (NNN)**: Correlations between next-adjacent sites (relevant for J1J2 model)

In [ ]:
# Set truncation parameters for correlation calculations
correlation_check_mw = 20
correlation_check_mac = 1e-7

println("="^80)
println("CORRELATION ISOTROPY CHECK CONFIGURATION")
println("="^80)
println("Truncation for observable propagation:")
println("  max_weight: $correlation_check_mw")
println("  min_abs_coeff: $correlation_check_mac")
println("="^80)

In [ ]:
"""
    calculate_nn_correlations_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)

Calculate nearest-neighbor spin-spin correlations in all three directions (X, Y, Z) 
at the middle of the chain.

For a chain with nq sites, computes correlations between sites i and i+1 where i = nq÷2.

Returns:
- C_xx: ⟨S_i^x S_{i+1}^x⟩
- C_yy: ⟨S_i^y S_{i+1}^y⟩
- C_zz: ⟨S_i^z S_{i+1}^z⟩
"""
function calculate_nn_correlations_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)
    # Choose middle site for correlations
    middle_site = nq ÷ 2
    site_i = middle_site
    site_j = middle_site + 1
    
    # Build correlation operators for nearest neighbors
    # Note: For spin-1/2, S^α = (1/2) σ^α, so S_i S_j = (1/4) σ_i σ_j
    # We'll compute σ_i σ_j and divide by 4 to get spin correlations
    
    C_xx = PauliSum(nq)
    C_yy = PauliSum(nq)
    C_zz = PauliSum(nq)
    
    # σ_i^x σ_{i+1}^x
    add!(C_xx, [:X, :X], [site_i, site_j])
    
    # σ_i^y σ_{i+1}^y
    add!(C_yy, [:Y, :Y], [site_i, site_j])
    
    # σ_i^z σ_{i+1}^z
    add!(C_zz, [:Z, :Z], [site_i, site_j])
    
    # Propagate observables through circuit
    C_xx_prop = propagate!(circuit, C_xx, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    C_yy_prop = propagate!(circuit, C_yy, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    C_zz_prop = propagate!(circuit, C_zz, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    
    # Calculate expectation values
    corr_xx = overlap_function(C_xx_prop, nq) / 4.0  # Factor of 1/4 for spin-1/2
    corr_yy = overlap_function(C_yy_prop, nq) / 4.0
    corr_zz = overlap_function(C_zz_prop, nq) / 4.0
    
    return corr_xx, corr_yy, corr_zz, site_i, site_j
end

"""
    calculate_nnn_correlations_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)

Calculate next-nearest-neighbor spin-spin correlations in all three directions (X, Y, Z)
at the middle of the chain.

For a chain with nq sites, computes correlations between sites i and i+2 where i = nq÷2.

Returns:
- C_xx: ⟨S_i^x S_{i+2}^x⟩
- C_yy: ⟨S_i^y S_{i+2}^y⟩
- C_zz: ⟨S_i^z S_{i+2}^z⟩
"""
function calculate_nnn_correlations_xyz(circuit, thetas, nq; max_weight=Inf, min_abs_coeff=0.0)
    # Choose middle site for correlations
    middle_site = nq ÷ 2
    site_i = middle_site
    site_j = middle_site + 2  # Next-nearest neighbor
    
    # Check if NNN exists (need at least 3 sites)
    if site_j > nq
        return NaN, NaN, NaN, site_i, site_j
    end
    
    # Build correlation operators for next-nearest neighbors
    C_xx = PauliSum(nq)
    C_yy = PauliSum(nq)
    C_zz = PauliSum(nq)
    
    # σ_i^x σ_{i+2}^x
    add!(C_xx, [:X, :X], [site_i, site_j])
    
    # σ_i^y σ_{i+2}^y
    add!(C_yy, [:Y, :Y], [site_i, site_j])
    
    # σ_i^z σ_{i+2}^z
    add!(C_zz, [:Z, :Z], [site_i, site_j])
    
    # Propagate observables through circuit
    C_xx_prop = propagate!(circuit, C_xx, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    C_yy_prop = propagate!(circuit, C_yy, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    C_zz_prop = propagate!(circuit, C_zz, thetas; min_abs_coeff=min_abs_coeff, max_weight=max_weight)
    
    # Calculate expectation values
    corr_xx = overlap_function(C_xx_prop, nq) / 4.0  # Factor of 1/4 for spin-1/2
    corr_yy = overlap_function(C_yy_prop, nq) / 4.0
    corr_zz = overlap_function(C_zz_prop, nq) / 4.0
    
    return corr_xx, corr_yy, corr_zz, site_i, site_j
end

println("✓ Correlation calculation functions defined");

### Nearest-Neighbor Correlations

In [ ]:
# Calculate NN correlations for all systems
nn_correlation_results = Dict{Int, Tuple{Float64, Float64, Float64, Int, Int}}()

all_nq_values = sort(collect(keys(hamiltonian_loop_runs)))

println("\n" * "="^80)
println("CALCULATING NEAREST-NEIGHBOR CORRELATIONS AT CHAIN CENTER")
println("="^80)
println("System sizes: $all_nq_values")
println("Truncation: max_weight=$correlation_check_mw, min_abs_coeff=$correlation_check_mac")
println("="^80)

for nq in all_nq_values
    println("\n--- Processing nq = $nq ---")
    
    loop_summary = hamiltonian_loop_runs[nq]
    
    # Get circuit and optimized parameters
    circuit = loop_summary["circuit"]
    thetas = loop_summary["optimized_thetas"]
    
    try
        # Calculate NN correlations in all three directions
        Cxx, Cyy, Czz, site_i, site_j = calculate_nn_correlations_xyz(circuit, thetas, nq; 
                                                                        max_weight=correlation_check_mw, 
                                                                        min_abs_coeff=correlation_check_mac)
        
        # Store results
        nn_correlation_results[nq] = (Cxx, Cyy, Czz, site_i, site_j)
        
        @printf "  Sites: %d - %d (middle of chain)\n" site_i site_j
        @printf "  C_xx = %+.8f\n" Cxx
        @printf "  C_yy = %+.8f\n" Cyy
        @printf "  C_zz = %+.8f\n" Czz
        @printf "  Isotropy violations:\n"
        @printf "    |C_xx - C_yy| = %.8e\n" abs(Cxx - Cyy)
        @printf "    |C_xx - C_zz| = %.8e\n" abs(Cxx - Czz)
        @printf "    |C_yy - C_zz| = %.8e\n" abs(Cyy - Czz)
        
    catch e
        @warn "Failed to calculate NN correlations for nq=$nq: $e"
    end
end

println("\n" * "="^80)
println("NN correlation calculation complete for $(length(nn_correlation_results)) systems")
println("="^80)

In [ ]:
# Plot NN correlation isotropy violations (differences) vs nq
if !isempty(nn_correlation_results)
    sorted_nqs = sort(collect(keys(nn_correlation_results)))
    
    # Calculate correlation differences (should be zero for isotropic correlations)
    # Replace values smaller than 1e-16 with machine precision floor for plotting
    machine_precision = 1e-16
    diff_xx_yy = [abs(nn_correlation_results[nq][1] - nn_correlation_results[nq][2]) for nq in sorted_nqs]
    diff_xx_zz = [abs(nn_correlation_results[nq][1] - nn_correlation_results[nq][3]) for nq in sorted_nqs]
    diff_yy_zz = [abs(nn_correlation_results[nq][2] - nn_correlation_results[nq][3]) for nq in sorted_nqs]
    
    # Replace values below machine precision with machine precision for log plot
    diff_xx_yy_plot = [val < machine_precision ? machine_precision : val for val in diff_xx_yy]
    diff_xx_zz_plot = [val < machine_precision ? machine_precision : val for val in diff_xx_zz]
    diff_yy_zz_plot = [val < machine_precision ? machine_precision : val for val in diff_yy_zz]
    
    title_text ="NN Correlation Isotropy Violations (Chain Center) - $HAMILTONIAN_NAME"      
    p_nn_diff = plot(
        xlabel="Number of Qubits",
        ylabel="Absolute Error",
        #xlim = [4,70],
        yscale=:log10,
        legend=:topright,
        grid=true,
        size=small,
        framestyle=:box,
        left_margin=8Plots.mm,
        bottom_margin=6Plots.mm
    )
    
    # Plot correlation differences
    plot!(p_nn_diff, sorted_nqs, diff_xx_yy_plot,
          label=L"|\langle S^x_i S^x_{i+1} \rangle - \langle S^y_i S^y_{i+1} \rangle|",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[1])
    
    plot!(p_nn_diff, sorted_nqs, diff_xx_zz_plot,
          label=L"|\langle S^x_i S^x_{i+1} \rangle - \langle S^z_i S^z_{i+1} \rangle|",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[2])
    
    plot!(p_nn_diff, sorted_nqs, diff_yy_zz_plot,
          label=L"|\langle S^y_i S^y_{i+1} \rangle-  \langle S^z_i S^z_{i+1} \rangle|",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[3])
    
    # Add horizontal line at machine precision for reference
    hline!(p_nn_diff, [machine_precision], 
           linestyle=:dash, 
           linewidth=1, 
           color=:gray, 
           label=L"\textrm{Machine precision} \ (10^{-16})",
           alpha=0.5)
    
    display(p_nn_diff)
end
save_thesis_plot(p_nn_diff,title_text)

### Next-Nearest-Neighbor Correlations (for J1J2)

# Thread Count Analysis: Peak RSS Memory

Compare peak_rss_mb across different thread counts to find the optimal thread configuration.

In [ ]:
# Load thread comparison data from threads_comp directory
tile_nq = 2
threads_comp_dir = "cluster_data/Benchmark_SystemHamiltonian(:$(HAMILTONIAN_NAME))_forwarddiff_grad_(max_weight = Inf,)_tile_nq_2_threads_comp_large"

println("="^80)
println("LOADING THREAD COMPARISON DATA")
println("="^80)
println("Directory: $threads_comp_dir")

function load_thread_comparison_data(dir_path; min_nq=4, max_nq=100)
    """
    Load data from thread comparison runs and organize by nq and n_threads.
    Returns: Dict{nq => Dict{n_threads => loop_summary}}
    """
    thread_data = Dict{Int, Dict{Int, Dict{String, Any}}}()
    
    if !isdir(dir_path)
        @warn "Directory does not exist: $dir_path"
        return thread_data
    end
    
    files = readdir(dir_path, join=true)
    log_files = filter(f -> endswith(f, ".jld2") && contains(f, "master_log_loop"), files)
    
    println("\nFound $(length(log_files)) log files")
    
    for filepath in log_files
        filename = basename(filepath)
        
        # Extract nq from filename
        nq_match = match(r"scaled(\d+)", filename)
        if isnothing(nq_match)
            continue
        end
        file_scaled_nq = parse(Int, nq_match.captures[1])
        
        # Skip if outside requested range
        if file_scaled_nq < min_nq || file_scaled_nq > max_nq
            continue
        end
        
        try
            data = JLD2.load(filepath)
            
            if haskey(data, "all_results")
                all_results = data["all_results"]
                
                # Use last (most accurate) run
                run_idx = length(all_results)
                selected_run = all_results[run_idx]
                
                if haskey(selected_run, "loop_summary")
                    loop_summary = selected_run["loop_summary"]
                    n_threads = get(loop_summary, "n_threads", 1)
                    peak_rss_mb = get(loop_summary, "peak_rss_mb", NaN)
                    
                    # Initialize nested dict if needed
                    if !haskey(thread_data, file_scaled_nq)
                        thread_data[file_scaled_nq] = Dict{Int, Dict{String, Any}}()
                    end
                    
                    # Store loop_summary for this nq and thread count
                    thread_data[file_scaled_nq][n_threads] = loop_summary
                    
                    println("  ✓ Loaded nq=$file_scaled_nq, n_threads=$n_threads, peak_rss_mb=$peak_rss_mb")
                end
            end
        catch e
            @warn "Failed to load $filename: $e"
        end
    end
    
    return thread_data
end
min_nq = 70
# Load the data
thread_comparison_data = load_thread_comparison_data(threads_comp_dir; min_nq=min_nq, max_nq=max_nq)

# Print summary
println("\n" * "="^80)
println("LOADED THREAD COMPARISON SUMMARY")
println("="^80)
for nq in sort(collect(keys(thread_comparison_data)))
    thread_counts = sort(collect(keys(thread_comparison_data[nq])))
    println("nq=$nq: thread counts = $thread_counts")
end
println("="^80)

In [ ]:
# Print num_reruns from a threaded run
if !isempty(thread_comparison_data)
    # Get first available nq
    first_nq = first(sort(collect(keys(thread_comparison_data))))
    # Get first available thread count for that nq
    first_thread = first(sort(collect(keys(thread_comparison_data[first_nq]))))
    # Get the loop_summary
    loop_summary = thread_comparison_data[first_nq][first_thread]
    
    if haskey(loop_summary, "num_reruns")
        num_reruns = loop_summary["num_reruns"]
        println("num_reruns for threaded runs (nq=$first_nq, threads=$first_thread): $num_reruns")
    else
        println("num_reruns not found in loop_summary for nq=$first_nq, threads=$first_thread")
        println("Available keys: $(keys(loop_summary))")
    end
else
    println("No thread comparison data loaded")
end

In [ ]:
# Extract peak_rss_mb for each nq and thread count
println("\n" * "="^80)
println("PEAK RSS MEMORY BY THREAD COUNT")
println("="^80)

# Organize data: nq => (thread_counts, peak_rss_values)
peak_rss_by_nq = Dict{Int, Tuple{Vector{Int}, Vector{Float64}}}()

for nq in sort(collect(keys(thread_comparison_data)))
    thread_dict = thread_comparison_data[nq]
    thread_counts = sort(collect(keys(thread_dict)))
    peak_rss_values = Float64[]
    
    println("\nnq = $nq:")
    println("-"^40)
    @printf "%-15s %-15s\n" "n_threads" "peak_rss_mb"
    println("-"^40)
    
    for n_threads in thread_counts
        loop_summary = thread_dict[n_threads]
        peak_rss_mb = get(loop_summary, "peak_rss_mb", NaN)
        push!(peak_rss_values, peak_rss_mb)
        
        @printf "%-15d %-15.2f\n" n_threads peak_rss_mb
    end
    
    peak_rss_by_nq[nq] = (thread_counts, peak_rss_values)
end

println("\n" * "="^80)

In [ ]:
# Extract total allocated memory (total_memory_gb) for each nq and thread count
println("\n" * "="^80)
println("TOTAL ALLOCATED MEMORY BY THREAD COUNT")
println("="^80)

# Organize data: nq => (thread_counts, total_memory_gb_values)
total_memory_by_nq = Dict{Int, Tuple{Vector{Int}, Vector{Float64}}}()

for nq in sort(collect(keys(thread_comparison_data)))
    thread_dict = thread_comparison_data[nq]
    thread_counts = sort(collect(keys(thread_dict)))
    total_memory_gb_values = Float64[]
    
    println("\nnq = $nq:")
    println("-"^40)
    @printf "%-15s %-20s\n" "n_threads" "total_memory_gb"
    println("-"^40)
    
    for n_threads in thread_counts
        loop_summary = thread_dict[n_threads]
        total_memory_gb = get(loop_summary, "total_memory_gb", NaN)
        push!(total_memory_gb_values, total_memory_gb)
        
        @printf "%-15d %-20.4f\n" n_threads total_memory_gb
    end
    
    total_memory_by_nq[nq] = (thread_counts, total_memory_gb_values)
end

println("\n" * "="^80)

In [ ]:
# Plot total allocated memory (GB) vs thread count for each nq
title_text = "Total Allocated Memory vs Thread Count - $HAMILTONIAN_NAME"

p_total_memory = plot(
    xlabel="Number of Threads",
    ylabel="Total Allocated Memory [GB]",
    legend=:topright,
    xticks = ([1,4,8,12]),
    grid=true,
    size=(600, 400),
    framestyle=:box,
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot each nq as a separate line
color_idx = 1
for nq in sort(collect(keys(total_memory_by_nq)))
    thread_counts, total_memory_gb_values = total_memory_by_nq[nq]
    
    plot!(p_total_memory, thread_counts, total_memory_gb_values,
          label="nq = $nq",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[mod1(color_idx, length(bold_hex))],
          markerstrokewidth=0)
    
    color_idx += 1
end

display(p_total_memory)
#save_thesis_plot(p_total_memory, title_text)

In [ ]:
# Combined analysis: Find optimal thread count based on both metrics
println("\n" * "="^80)
println("COMBINED MEMORY ANALYSIS (Peak RSS + Total Allocated)")
println("="^80)

for nq in sort(collect(keys(peak_rss_by_nq)))
    println("\nnq = $nq:")
    println("-"^60)
    
    thread_counts_rss, peak_rss_values = peak_rss_by_nq[nq]
    thread_counts_mem, total_memory_gb_values = total_memory_by_nq[nq]
    
    @printf "%-10s %-18s %-20s\n" "Threads" "Peak RSS (MB)" "Total Alloc (GB)"
    println("-"^60)
    
    for (i, n_threads) in enumerate(thread_counts_rss)
        @printf "%-10d %-18.2f %-20.4f\n" n_threads peak_rss_values[i] total_memory_gb_values[i]
    end
    
    # Find optimal (minimum) for both metrics
    min_rss_idx = argmin(peak_rss_values)
    min_mem_idx = argmin(total_memory_gb_values)
    
    println("-"^60)
    println("Optimal (min Peak RSS): $(thread_counts_rss[min_rss_idx]) threads")
    println("Optimal (min Total Alloc): $(thread_counts_mem[min_mem_idx]) threads")
    
    if thread_counts_rss[min_rss_idx] == thread_counts_mem[min_mem_idx]
        println("✓ Both metrics agree on optimal thread count!")
    else
        println("⚠ Metrics disagree - consider workload priorities")
    end
end

println("\n" * "="^80)

In [ ]:
# Plot peak_rss_mb vs thread count for each nq
title_text = "Peak RSS Memory vs Thread Count - $HAMILTONIAN_NAME"

p_threads = plot(
    xlabel="Number of Threads",
    ylabel="Peak RSS Memory [GB]",
    xticks = ([1,2,4,8,12]),
    legend=:bottomright,
    grid=true,
    size=(600, 400),
    framestyle=:box,
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot each nq as a separate line
color_idx = 1
for nq in sort(collect(keys(peak_rss_by_nq)))
    thread_counts, peak_rss_values = peak_rss_by_nq[nq]
    
    plot!(p_threads, thread_counts, peak_rss_values/1000,
          label="nq = $nq",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[mod1(color_idx, length(bold_hex))],
          markerstrokewidth=0)
    
    color_idx += 1
end

display(p_threads)
#save_thesis_plot(p_threads, title_text)

## Total Runtime Analysis

Compare total elapsed time across different thread counts to assess parallel efficiency.

In [ ]:
# Extract total elapsed time for each nq and thread count
println("\n" * "="^80)
println("TOTAL RUNTIME BY THREAD COUNT")
println("="^80)

# Organize data: nq => (thread_counts, runtime_seconds)
runtime_by_nq = Dict{Int, Tuple{Vector{Int}, Vector{Float64}}}()

for nq in sort(collect(keys(thread_comparison_data)))
    thread_dict = thread_comparison_data[nq]
    thread_counts = sort(collect(keys(thread_dict)))
    runtime_seconds = Float64[]
    
    println("\nnq = $nq:")
    println("-"^50)
    @printf "%-15s %-20s %-15s\n" "n_threads" "Runtime (s)" "Runtime (min)"
    println("-"^50)
    
    for n_threads in thread_counts
        loop_summary = thread_dict[n_threads]
        total_time_s = get(loop_summary, "total_elapsed_time_s", NaN)
        push!(runtime_seconds, total_time_s)
        
        @printf "%-15d %-20.2f %-15.2f\n" n_threads total_time_s (total_time_s/60)
    end
    
    runtime_by_nq[nq] = (thread_counts, runtime_seconds)
end

println("\n" * "="^80)

In [ ]:
# Plot total runtime vs thread count for each nq
title_text = "Total Runtime vs Thread Count - $HAMILTONIAN_NAME"
small = (600,400)
p_runtime = plot(
    xlabel="Number of Threads",
    ylabel="Total Runtime [min]",
    xticks = ([1,2,4,8,12]),
    legend=:topright,
    grid=true,
    size=small,
    framestyle=:box,
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot each nq as a separate line
color_idx = 1
for nq in sort(collect(keys(runtime_by_nq)))
    thread_counts, runtime_seconds = runtime_by_nq[nq]
    
    plot!(p_runtime, thread_counts, runtime_seconds/60,
          label="nq = $nq",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[mod1(color_idx, length(bold_hex))],
          markerstrokewidth=0)
    
    color_idx += 1
end

display(p_runtime)
#save_thesis_plot(p_runtime, title_text)

In [ ]:
# Calculate speedup and parallel efficiency
println("\n" * "="^80)
println("PARALLEL SPEEDUP & EFFICIENCY ANALYSIS")
println("="^80)

for nq in sort(collect(keys(runtime_by_nq)))
    thread_counts, runtime_seconds = runtime_by_nq[nq]
    
    # Use single-thread runtime as baseline
    if thread_counts[1] == 1
        baseline_time = runtime_seconds[1]
    else
        # If we don't have single-thread data, use the first available
        baseline_time = runtime_seconds[1]
        @warn "No single-thread baseline for nq=$nq, using $(thread_counts[1]) threads as baseline"
    end
    
    println("\nnq = $nq (baseline: $(round(baseline_time, digits=2)) s):")
    println("-"^70)
    @printf "%-12s %-15s %-15s %-20s\n" "n_threads" "Runtime (s)" "Speedup" "Efficiency (%)"
    println("-"^70)
    
    for (i, n_threads) in enumerate(thread_counts)
        time = runtime_seconds[i]
        speedup = baseline_time / time
        efficiency = (speedup / n_threads) * 100
        
        @printf "%-12d %-15.2f %-15.2f %-20.1f\n" n_threads time speedup efficiency
    end
    
    # Find optimal thread count (best speedup)
    speedups = [baseline_time / t for t in runtime_seconds]
    best_speedup_idx = argmax(speedups)
    best_n_threads = thread_counts[best_speedup_idx]
    
    println("-"^70)
    println("Best speedup: $(round(speedups[best_speedup_idx], digits=2))x at $best_n_threads threads")
end

println("\n" * "="^80)

In [ ]:
# Plot speedup curves
title_text = "Parallel Speedup vs Thread Count - $HAMILTONIAN_NAME"

p_speedup = plot(
    xlabel="Number of Threads",
    ylabel="Speedup Factor",
    legend=:topleft,
    grid=true,
    size=(600, 400),
    framestyle=:box,
    left_margin=10Plots.mm,
    bottom_margin=6Plots.mm
)

# Plot each nq as a separate line
color_idx = 1
for nq in sort(collect(keys(runtime_by_nq)))
    thread_counts, runtime_seconds = runtime_by_nq[nq]
    
    # Calculate speedup (using first thread count as baseline)
    baseline_time = runtime_seconds[1]
    speedups = [baseline_time / t for t in runtime_seconds]
    
    plot!(p_speedup, thread_counts, speedups,
          label="nq = $nq",
          marker=:circle,
          markersize=6,
          linewidth=2,
          color=bold_hex[mod1(color_idx, length(bold_hex))],
          markerstrokewidth=0)
    
    color_idx += 1
end

# Add ideal linear speedup reference line
if !isempty(runtime_by_nq)
    thread_counts_ref = collect(keys(thread_comparison_data[first(keys(runtime_by_nq))]))
    min_threads = minimum([minimum(first(runtime_by_nq[nq])) for nq in keys(runtime_by_nq)])
    max_threads = maximum([maximum(first(runtime_by_nq[nq])) for nq in keys(runtime_by_nq)])
    
    # Normalize to first thread count as baseline
    ideal_speedup = collect(min_threads:max_threads) ./ min_threads
    
    plot!(p_speedup, collect(min_threads:max_threads), ideal_speedup,
          label="Linear",
          linestyle=:dash,
          linewidth=2,
          color=:gray,
          alpha=0.5)
end

display(p_speedup)
#save_thesis_plot(p_speedup, title_text)

In [ ]:
# Print circuit depth for every system size and thread count
println("\n" * "="^80)
println("CIRCUIT DEPTH BY SYSTEM SIZE AND THREAD COUNT")
println("="^80)

for nq in sort(collect(keys(thread_comparison_data)))
    thread_dict = thread_comparison_data[nq]
    thread_counts = sort(collect(keys(thread_dict)))
    
    println("\nnq = $nq:")
    println("-"^50)
    @printf "%-15s %-15s\n" "n_threads" "Circuit Depth"
    println("-"^50)
    
    depths = Int[]
    for n_threads in thread_counts
        loop_summary = thread_dict[n_threads]
        depth = get(loop_summary, "final_circuit_depth", -1)
        push!(depths, depth)
        
        @printf "%-15d %-15d\n" n_threads depth
    end
    
    # Check if all depths are the same
    println("-"^50)
    if all(d == depths[1] for d in depths)
        println("✓ Circuit depth is CONSTANT at $(depths[1]) across all thread counts")
    else
        println("⚠ Circuit depth VARIES: min=$(minimum(depths)), max=$(maximum(depths))")
        println("  Depths: $depths")
    end
end

println("\n" * "="^80)
println("VERIFICATION SUMMARY")
println("="^80)

# Check if depths are constant across all nq and thread counts
all_constant = true
for nq in sort(collect(keys(thread_comparison_data)))
    thread_dict = thread_comparison_data[nq]
    thread_counts = sort(collect(keys(thread_dict)))
    
    depths = [get(thread_dict[n_threads], "final_circuit_depth", -1) for n_threads in thread_counts]
    
    if !all(d == depths[1] for d in depths)
        all_constant = false
        println("⚠ nq=$nq has varying depths across thread counts")
    end
end

if all_constant
    println("✓ ALL system sizes have constant circuit depth across thread counts")
    println("  This confirms that threading does NOT affect circuit construction!")
else
    println("⚠ Some system sizes have varying circuit depths")
    println("  This suggests threading may affect operator selection randomness")
end

println("="^80)